<a href="https://colab.research.google.com/github/eyacharfeddine/alpr-using-domain-adaptation-/blob/main/domain_adaptation_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/davidjaw/Domain-adaptation-object-detection-with-YOLOv8.git
%cd Domain-adaptation-object-detection-with-YOLOv8


Cloning into 'Domain-adaptation-object-detection-with-YOLOv8'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 37 (delta 13), reused 36 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 32.71 KiB | 16.35 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/Domain-adaptation-object-detection-with-YOLOv8


In [ ]:
!git clone https://github.com/ultralytics/ultralytics.git
%cd ultralytics


Cloning into 'ultralytics'...
remote: Enumerating objects: 56957, done.
remote: Counting objects: 100% (867/867), done.
remote: Compressing objects: 100% (403/403), done.
remote: Total 56957 (delta 727), reused 468 (delta 464), pack-reused 56090 (from 5)
Receiving objects: 100% (56957/56957), 31.75 MiB | 30.00 MiB/s, done.
Resolving deltas: 100% (42146/42146), done.
/content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics


In [ ]:
%cd Domain-adaptation-object-detection-with-YOLOv8

[Errno 2] No such file or directory: 'Domain-adaptation-object-detection-with-YOLOv8'
/content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics


In [ ]:
!pip install ultralytics


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install specific PyTorch and Torchvision versions
!pip install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118


Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 750.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 97.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 36.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lit: filename=lit-15.0.7-py3-none-any.whl size=89990 sha256=10cff4efbd655b3e2db82be455084f635d48ee5f6efa701614d602706797aefe
  Stored in directory: /root/.cache/pip/wheels/fc/5d/45/34fe9945d5e45e261134e72284395be36c2d4828af38e2b0fe
Successfully built lit
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
  Attempting uninstall: torch
    Found existing installation: torch 2.6.0+cu124
    Uninstalling torch-2.6.0+cu124:
      Successfully uninstalled torc

In [ ]:
!ls

data  LICENSE  README.md  scripts  train.py


In [ ]:
import os
print("Visible dataset exists:", os.path.exists('/content/drive/My Drive/pguard/dataset.yaml'))
print("Thermal dataset exists:", os.path.exists('/content/drive/My Drive/pguard/thermal/thermal.yaml'))
print("Pre-trained model exists:", os.path.exists('/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'))

Visible dataset exists: True
Thermal dataset exists: True
Pre-trained model exists: True


In [ ]:
%%writefile scripts/vis.py
import os
from utils import visualize, get_img_label_path
import cv2
import torch
import torchvision
import numpy as np

# Update the path to your thermal dataset
img_paths, label_paths = get_img_label_path('/content/drive/My Drive/pguard/thermal')

# Create a directory to save visualized images
os.makedirs('/content/thermal_visualizations', exist_ok=True)

# Visualize the first 5 images
for i, img_path in enumerate(img_paths[:5]):
    img = cv2.imread(img_path)
    labels = []
    with open(label_paths[i], 'r') as f:
        for line in f:
            labels.append(list(map(float, line.split(' '))))

    # Save the visualized image
    save_path = f'/content/thermal_visualizations/thermal_sample_{i}.jpg'
    visualize(img, labels, .7, save_path=save_path)
    print(f"Saved visualization to {save_path}")

Overwriting scripts/vis.py


In [ ]:
%%writefile scripts/utils.py
import numpy as np
import os
import cv2

def load_setup(yaml_path='config.yaml'):
    import yaml
    with open(yaml_path, 'r') as f:
        config = yaml.load(f, Loader=yaml.FullLoader)
    if os.name == 'nt':
        config = config['windows']
    else:
        config = config['linux']
    return config

def parse_setup(config: dict, add_worker=False):
    s = ''
    for key in config:
        s += f'--{key} {config[key]} '
    if add_worker:
        s += f'--workers {min(os.cpu_count(), 8) if os.name != "nt" else 3} '
    return s

def visualize(img: np.ndarray, labels: [[int | float]], ratio=1, save_path=None):
    color_palette = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255), (0, 255, 255)]
    thickness = int(max(img.shape) * 0.003) + 1
    font_scale = img.shape[1] / 1000.0

    if len(labels) > 0:
        for label in labels:
            c, x, y, w, h = label
            x = int(x * img.shape[1])
            y = int(y * img.shape[0])
            w = int(w * img.shape[1])
            h = int(h * img.shape[0])
            color = color_palette[int(c) % len(color_palette)]
            cv2.rectangle(img, (int(x - w / 2), int(y - h / 2)), (int(x + w / 2), int(y + h / 2)), color, thickness)
            text = str(int(c))
            (text_width, text_height), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
            overlay = img.copy()
            cv2.rectangle(overlay, (int(x - w / 2), int(y - h / 2) - text_height - 10),
                          (int(x - w / 2) + text_width, int(y - h / 2)), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.8, img, 0.2, 0, img)
            text_pos = (int(x - w / 2), int(y - h / 2) - 10)
            cv2.putText(img, text, text_pos, cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, thickness)

    if ratio != 1:
        img = cv2.resize(img, (int(img.shape[1] * ratio), int(img.shape[0] * ratio)))

    if save_path:
        cv2.imwrite(save_path, img)
    else:
        cv2.imshow('img', img)
        cv2.waitKey(0)

def restore_coord(img_shape: [int], x: float, y: float, w: float, h: float) -> (int, int, int, int):
    x = int(x * img_shape[1])
    y = int(y * img_shape[0])
    w = int(w * img_shape[1])
    h = int(h * img_shape[0])
    return x, y, w, h

def get_img_label_path(base_path) -> ([str], [str]):
    image_dir = os.path.join(base_path, "images")
    label_dir = os.path.join(base_path, "labels")
    img_files = sorted(os.listdir(image_dir))
    label_files = sorted(os.listdir(label_dir))
    img_paths = [os.path.join(image_dir, img_file) for img_file in img_files]
    label_paths = [os.path.join(label_dir, label_file) for label_file in label_files]
    return img_paths, label_paths

def crop_object(img: np.ndarray, x, y, w, h, target_size=None) -> np.ndarray:
    x, y, w, h = restore_coord(img.shape, x, y, w, h)
    img = img[y - h // 2:y + h // 2, x - w // 2:x + w // 2]
    if target_size is not None:
        img = cv2.resize(img, target_size)
    return img

def get_object_mask(img_shape, x, y, w, h) -> np.ndarray:
    x, y, w, h = restore_coord(img_shape, x, y, w, h)
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    mask[y - h // 2:y + h // 2, x - w // 2:x + w // 2] = 255
    return mask

def tsne_2d(features, labels, save_path, pca_n=100, max_num=2500, name_list=None, show_legend=True, fn=None, title=None):
    from sklearn.manifold import TSNE
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import os

    print(f'Performing PCA + t-SNE on {features.shape[0]} features...')
    print(f'- Processing PCA with {pca_n} components...', end='')
    if features.shape[0] < pca_n:
        print(f'Warning: features.shape[0] ({features.shape[0]}) < pca_n ({pca_n}),'
              f'Setting pca_n to features.shape[0] // 2 ({features.shape[0] // 2})')
        pca_n = features.shape[0] // 2
    pca = PCA(n_components=pca_n)
    reduced_features = pca.fit_transform(features)
    print('Done.')

    if reduced_features.shape[0] > max_num:
        print(f'Warning: features.shape[0] ({reduced_features.shape[0]}) > max_num ({max_num}),'
              f'only plotting the first {max_num} features to avoid memory error.')
        indices = np.random.choice(reduced_features.shape[0], max_num, replace=False)
        reduced_features = reduced_features[indices]
        labels = labels[indices]

    num_classes = len(np.unique(labels))
    palette = sns.color_palette("husl", num_classes)

    if name_list:
        if len(name_list) != num_classes:
            raise ValueError(f'len(name_list) ({len(name_list)}) != num_classes ({num_classes})')
    else:
        name_list = [f'Class {i}' for i in range(num_classes)]

    print(f'- Processing t-SNE with {num_classes} classes...', end='')
    tsne = TSNE(n_components=2, random_state=42)
    transformed_data = tsne.fit_transform(reduced_features[:max_num, :])
    labels_array = labels[:max_num]
    print('Done.')

    fig, ax = plt.subplots(figsize=(11, 10), dpi=100)
    for i in range(num_classes):
        plt.scatter(transformed_data[labels_array == i, 0], transformed_data[labels_array == i, 1], label=name_list[i],
                    color=palette[i], alpha=0.75)

    if show_legend:
        plt.legend()
    if title is None:
        title = f't-SNE Clustering ({os.path.basename(save_path)})'
    plt.title(title)
    if fn is None:
        fn = f'{os.path.basename(save_path)}-PCA-tSNE'
    plt.savefig(f'{save_path}/{fn}.png')
    plt.clf()
    plt.close()

def tsne_2d_on_dir(dir_name, pca_dim=100, name_list=None):
    features_list = []
    labels = []

    for file_name in os.listdir(dir_name):
        if file_name.endswith('.npy'):
            file_path = os.path.join(dir_name, file_name)
            features = np.load(file_path)
            batch_size = features.shape[0]
            class_idx, _ = file_name.split('_')
            features_list.append(features)
            labels += [int(class_idx)] * batch_size

    features_array = np.vstack(features_list)
    labels_array = np.array(labels)
    if len(features_array.shape) > 2:
        features_array = features_array.reshape(features_array.shape[0], -1)
    tsne_2d(features_array, labels_array, dir_name, pca_n=pca_dim, name_list=name_list)

def tsne_clustering_3d(dir_name):
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    import seaborn as sns
    from mpl_toolkits.mplot3d import Axes3D
    features_list = []
    labels = []

    for file_name in os.listdir(dir_name):
        if file_name.endswith('.npy'):
            file_path = os.path.join(dir_name, file_name)
            features = np.load(file_path)
            batch_size = features.shape[0]
            class_idx, _ = file_name.split('_')
            features_list.append(features)
            labels += [int(class_idx)] * batch_size

    features_array = np.vstack(features_list)
    labels_array = np.array(labels)

    num_classes = len(np.unique(labels_array))
    palette = sns.color_palette("Set2", num_classes)

    tsne = TSNE(n_components=3, random_state=42)
    transformed_data = tsne.fit_transform(features_array[:2800, :])
    labels_array = labels_array[:2800]

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    for i in range(num_classes):
        ax.scatter(transformed_data[labels_array == i, 0],
                   transformed_data[labels_array == i, 1],
                   transformed_data[labels_array == i, 2],
                   label=f"Class {i}",
                   color=palette[i])

    ax.legend()
    ax.set_title(f't-SNE Clustering in 3D ({os.path.basename(dir_name)})')
    plt.savefig(f'{os.path.dirname(dir_name)}/{os.path.basename(dir_name)}-tSNE-3D.png')
    plt.clf()
    plt.close()

Overwriting scripts/utils.py


In [ ]:
%cd /content/Domain-adaptation-object-detection-with-YOLOv8/scripts
!python vis.py

/content/Domain-adaptation-object-detection-with-YOLOv8/scripts

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/content/Domain-adaptation-object-detection-with-YOLOv8/scripts/vis.py", line 5, in <module>
    import torchvision
  File "/usr/local/lib/python3.11/dist-packages/torchvision/__init__.py", line 6, in <module>
    from torchvision import datasets, io, models, ops, transforms, utils
  File "/usr/local/lib/python3.11/dist-packages/torchvision/models/__init__.py", line 17, in <module>
    from . import detection, optical_flow, quantization, segmentation, 

In [ ]:
from google.colab import files
import os

for i in range(5):
    file_path = f'/content/thermal_visualizations/thermal_sample_{i}.jpg'
    if os.path.exists(file_path):
        files.download(file_path)

In [ ]:
import shutil
shutil.copytree('/content/thermal_visualizations', '/content/drive/My Drive/pguard/thermal_visualizations', dirs_exist_ok=True)
print("Visualizations saved to /content/drive/My Drive/pguard/thermal_visualizations")

In [ ]:

# Downgrade NumPy to a compatible version
!pip install numpy==1.26.4

In [ ]:
# Modify vis.py
%%writefile scripts/vis.py
import os
from utils import visualize, get_img_label_path
import cv2
import torch
import torchvision
import numpy as np

# Update the path to your thermal dataset
img_paths, label_paths = get_img_label_path('/content/drive/My Drive/pguard/thermal')

# Create a directory to save visualized images
os.makedirs('/content/thermal_visualizations', exist_ok=True)

# Visualize the first 5 images
for i, img_path in enumerate(img_paths[:5]):
    img = cv2.imread(img_path)
    labels = []
    with open(label_paths[i], 'r') as f:
        for line in f:
            labels.append(list(map(float, line.split(' '))))

    # Save the visualized image
    save_path = f'/content/thermal_visualizations/thermal_sample_{i}.jpg'
    visualize(img, labels, .7, save_path=save_path)
    print(f"Saved visualization to {save_path}")

# Modify utils.py
%%writefile scripts/utils.py
import numpy as np
import os
import cv2

def load_setup(yaml_path='config.yaml'):
    import yaml
    with open(yaml_path, 'r') as f:
        config = yaml.load(f, Loader=yaml.FullLoader)
    if os.name == 'nt':
        config = config['windows']
    else:
        config = config['linux']
    return config

def parse_setup(config: dict, add_worker=False):
    s = ''
    for key in config:
        s += f'--{key} {config[key]} '
    if add_worker:
        s += f'--workers {min(os.cpu_count(), 8) if os.name != "nt" else 3} '
    return s

def visualize(img: np.ndarray, labels: [[int | float]], ratio=1, save_path=None):
    color_palette = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255), (0, 255, 255)]
    thickness = int(max(img.shape) * 0.003) + 1
    font_scale = img.shape[1] / 1000.0

    if len(labels) > 0:
        for label in labels:
            c, x, y, w, h = label
            x = int(x * img.shape[1])
            y = int(y * img.shape[0])
            w = int(w * img.shape[1])
            h = int(h * img.shape[0])
            color = color_palette[int(c) % len(color_palette)]
            cv2.rectangle(img, (int(x - w / 2), int(y - h / 2)), (int(x + w / 2), int(y + h / 2)), color, thickness)
            text = str(int(c))
            (text_width, text_height), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
            overlay = img.copy()
            cv2.rectangle(overlay, (int(x - w / 2), int(y - h / 2) - text_height - 10),
                          (int(x - w / 2) + text_width, int(y - h / 2)), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.8, img, 0.2, 0, img)
            text_pos = (int(x - w / 2), int(y - h / 2) - 10)
            cv2.putText(img, text, text_pos, cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, thickness)

    if ratio != 1:
        img = cv2.resize(img, (int(img.shape[1] * ratio), int(img.shape[0] * ratio)))

    if save_path:
        cv2.imwrite(save_path, img)
    else:
        cv2.imshow('img', img)
        cv2.waitKey(0)

def restore_coord(img_shape: [int], x: float, y: float, w: float, h: float) -> (int, int, int, int):
    x = int(x * img_shape[1])
    y = int(y * img_shape[0])
    w = int(w * img_shape[1])
    h = int(h * img_shape[0])
    return x, y, w, h

def get_img_label_path(base_path) -> ([str], [str]):
    image_dir = os.path.join(base_path, "images")
    label_dir = os.path.join(base_path, "labels")
    img_files = sorted(os.listdir(image_dir))
    label_files = sorted(os.listdir(label_dir))
    img_paths = [os.path.join(image_dir, img_file) for img_file in img_files]
    label_paths = [os.path.join(label_dir, label_file) for label_file in label_files]
    return img_paths, label_paths

def crop_object(img: np.ndarray, x, y, w, h, target_size=None) -> np.ndarray:
    x, y, w, h = restore_coord(img.shape, x, y, w, h)
    img = img[y - h // 2:y + h // 2, x - w // 2:x + w // 2]
    if target_size is not None:
        img = cv2.resize(img, target_size)
    return img

def get_object_mask(img_shape, x, y, w, h) -> np.ndarray:
    x, y, w, h = restore_coord(img_shape, x, y, w, h)
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    mask[y - h // 2:y + h // 2, x - w // 2:x + w // 2] = 255
    return mask

def tsne_2d(features, labels, save_path, pca_n=100, max_num=2500, name_list=None, show_legend=True, fn=None, title=None):
    from sklearn.manifold import TSNE
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import os

    print(f'Performing PCA + t-SNE on {features.shape[0]} features...')
    print(f'- Processing PCA with {pca_n} components...', end='')
    if features.shape[0] < pca_n:
        print(f'Warning: features.shape[0] ({features.shape[0]}) < pca_n ({pca_n}),'
              f'Setting pca_n to features.shape[0] // 2 ({features.shape[0] // 2})')
        pca_n = features.shape[0] // 2
    pca = PCA(n_components=pca_n)
    reduced_features = pca.fit_transform(features)
    print('Done.')

    if reduced_features.shape[0] > max_num:
        print(f'Warning: features.shape[0] ({reduced_features.shape[0]}) > max_num ({max_num}),'
              f'only plotting the first {max_num} features to avoid memory error.')
        indices = np.random.choice(reduced_features.shape[0], max_num, replace=False)
        reduced_features = reduced_features[indices]
        labels = labels[indices]

    num_classes = len(np.unique(labels))
    palette = sns.color_palette("husl", num_classes)

    if name_list:
        if len(name_list) != num_classes:
            raise ValueError(f'len(name_list) ({len(name_list)}) != num_classes ({num_classes})')
    else:
        name_list = [f'Class {i}' for i in range(num_classes)]

    print(f'- Processing t-SNE with {num_classes} classes...', end='')
    tsne = TSNE(n_components=2, random_state=42)
    transformed_data = tsne.fit_transform(reduced_features[:max_num, :])
    labels_array = labels[:max_num]
    print('Done.')

    fig, ax = plt.subplots(figsize=(11, 10), dpi=100)
    for i in range(num_classes):
        plt.scatter(transformed_data[labels_array == i, 0], transformed_data[labels_array == i, 1], label=name_list[i],
                    color=palette[i], alpha=0.75)

    if show_legend:
        plt.legend()
    if title is None:
        title = f't-SNE Clustering ({os.path.basename(save_path)})'
    plt.title(title)
    if fn is None:
        fn = f'{os.path.basename(save_path)}-PCA-tSNE'
    plt.savefig(f'{save_path}/{fn}.png')
    plt.clf()
    plt.close()

def tsne_2d_on_dir(dir_name, pca_dim=100, name_list=None):
    features_list = []
    labels = []

    for file_name in os.listdir(dir_name):
        if file_name.endswith('.npy'):
            file_path = os.path.join(dir_name, file_name)
            features = np.load(file_path)
            batch_size = features.shape[0]
            class_idx, _ = file_name.split('_')
            features_list.append(features)
            labels += [int(class_idx)] * batch_size

    features_array = np.vstack(features_list)
    labels_array = np.array(labels)
    if len(features_array.shape) > 2:
        features_array = features_array.reshape(features_array.shape[0], -1)
    tsne_2d(features_array, labels_array, dir_name, pca_n=pca_dim, name_list=name_list)

def tsne_clustering_3d(dir_name):
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    import seaborn as sns
    from mpl_toolkits.mplot3d import Axes3D
    features_list = []
    labels = []

    for file_name in os.listdir(dir_name):
        if file_name.endswith('.npy'):
            file_path = os.path.join(dir_name, file_name)
            features = np.load(file_path)
            batch_size = features.shape[0]
            class_idx, _ = file_name.split('_')
            features_list.append(features)
            labels += [int(class_idx)] * batch_size

    features_array = np.vstack(features_list)
    labels_array = np.array(labels)

    num_classes = len(np.unique(labels_array))
    palette = sns.color_palette("Set2", num_classes)

    tsne = TSNE(n_components=3, random_state=42)
    transformed_data = tsne.fit_transform(features_array[:2800, :])
    labels_array = labels_array[:2800]

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    for i in range(num_classes):
        ax.scatter(transformed_data[labels_array == i, 0],
                   transformed_data[labels_array == i, 1],
                   transformed_data[labels_array == i, 2],
                   label=f"Class {i}",
                   color=palette[i])

    ax.legend()
    ax.set_title(f't-SNE Clustering in 3D ({os.path.basename(dir_name)})')
    plt.savefig(f'{os.path.dirname(dir_name)}/{os.path.basename(dir_name)}-tSNE-3D.png')
    plt.clf()
    plt.close()

# Run vis.py to visualize the thermal dataset
%cd /content/Domain-adaptation-object-detection-with-YOLOv8/scripts
!python vis.py

# Download the visualizations
from google.colab import files
import os

for i in range(5):
    file_path = f'/content/thermal_visualizations/thermal_sample_{i}.jpg'
    if os.path.exists(file_path):
        files.download(file_path)

# Save visualizations to Google Drive
import shutil
shutil.copytree('/content/thermal_visualizations', '/content/drive/My Drive/pguard/thermal_visualizations', dirs_exist_ok=True)
print("Visualizations saved to /content/drive/My Drive/pguard/thermal_visualizations")

Overwriting scripts/vis.py


In [ ]:
%%writefile train.py
from ultralytics.models import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr, emojis, clean_url
import ultralytics.utils.callbacks.tensorboard as tb_module
import numpy as np
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from functools import partial
import random
import os

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super(GradientReversalLayer, self).__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h // 2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim_h // 2, dim_h // 2),
                nn.LeakyReLU(0.2, inplace=True),
            ) for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h // 2 * 4, dim_h // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h // 2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        x = x.split(x.shape[1] // 2, dim=1)
        xs = [x[0]]
        for m in self.neck:
            x = m(x[1]) if isinstance(x, tuple) else m(x)
            xs.append(x)
        x = torch.cat(xs, dim=1)
        return self.head(x)

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        if chs is None:
            chs = [64, 128, 256]
            self.chs = chs
            self.f_len = len(chs)
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.p = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i == 0 else chs[i] * 2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64),
                nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(32),
                nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i + 1] if i + 1 < len(chs) else chs[i], kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(chs[i + 1] if i + 1 < len(chs) else chs[i]),
                nn.SiLU(inplace=True),
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, fs: list[torch.tensor]):
        with torch.cuda.amp.autocast(self.amp):
            assert len(fs) == self.f_len, f'Expected {self.f_len} feature maps, got {len(fs)}'
            fs = [self.grl(f) for f in fs]
            x = self.p[0](fs[0])
            for i in range(1, len(fs)):
                x = torch.cat((x, fs[i]), dim=1)
                x = self.p[i](x)
            return self.head(x)

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, *args, **kwargs):
        super(CustomTrainer, self).__init__(*args, **kwargs)
        # Load the target domain dataset (thermal)
        try:
            if target_domain_data_cfg.endswith(('yaml', 'yml')) or self.args.task in ('detect', 'segment', 'pose'):
                self.t_data = check_det_dataset(target_domain_data_cfg)
                # Initialize target datasets directly in __init__
                self.t_trainset, self.t_testset = super().get_dataset()
        except Exception as e:
            raise RuntimeError(emojis(f"Dataset '{clean_url(target_domain_data_cfg)}' error ❌ {e}")) from e
        self.t_iter = None
        self.t_train_loader = None
        self.model_hook_handler = []
        self.model_hook_layer_idx: list[int] = [2, 4, 6]
        self.roi_size = list(reversed([20 * 2 ** i for i in range(len(self.model_hook_layer_idx))]))
        self.model_hooked_features: None | list[torch.tensor] = None
        self.discriminator_model = None
        self.projection_model = None
        self.additional_models = []
        self.tensorboard_writer = None  # Initialize TensorBoard writer
        self.add_callback('on_train_start', self.init_helper_model)
        self.add_callback('on_train_start', self.init_tensorboard_writer)  # Add callback to initialize TensorBoard writer
        self.add_callback('teardown', self.close_tensorboard_writer)  # Add callback to close TensorBoard writer

    def init_helper_model(self, *args, **kwargs):
        self.discriminator_model = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator_model)

    def init_tensorboard_writer(self, *args, **kwargs):
        if RANK in (-1, 0):  # Only initialize on the main process
            self.tensorboard_writer = tb_module.SummaryWriter(log_dir=self.save_dir)  # Use tb_module.SummaryWriter
            LOGGER.info(f"TensorBoard writer initialized at {self.save_dir}")

    def close_tensorboard_writer(self, *args, **kwargs):
        if self.tensorboard_writer is not None:
            self.tensorboard_writer.close()
            LOGGER.info("TensorBoard writer closed")

    def get_t_batch(self):
        if self.t_iter is None:
            self.t_train_loader = self.get_dataloader(self.t_trainset, batch_size=self.batch_size, rank=RANK, mode='train')
            self.t_iter = iter(self.t_train_loader)
        try:
            batch = next(self.t_iter)
        except StopIteration:
            self.t_iter = iter(self.t_train_loader)
            batch = next(self.t_iter)
        return batch

    def activate_hook(self, layer_indices: list[int] = None):
        if layer_indices is not None:
            self.model_hook_layer_idx = layer_indices
        self.model_hooked_features = [None for _ in self.model_hook_layer_idx]
        self.model_hook_handler = \
            [self.model.model[l].register_forward_hook(self.hook_fn(i)) for i, l in enumerate(self.model_hook_layer_idx)]

    def deactivate_hook(self):
        if self.model_hook_handler is not None:
            for hook in self.model_hook_handler:
                hook.remove()
            self.model_hooked_features = None
            self.model_hook_handler = []

    def hook_fn(self, hook_idx):
        def hook(m, i, o):
            self.model_hooked_features[hook_idx] = o
        return hook

    def get_dis_output_from_hooked_features(self, batch):
        if self.model_hooked_features is not None:
            bbox_batch_idx = batch['batch_idx'].unsqueeze(-1)
            bbox = batch['bboxes']
            bbox = box_convert(bbox, 'cxcywh', 'xyxy')
            rois = []
            for fidx, f in enumerate(self.model_hooked_features):
                f_bbox = bbox * f.shape[-1]
                f_bbox = torch.cat((bbox_batch_idx, f_bbox), dim=-1)
                f_roi = roi_align(f, f_bbox.to(f.device), output_size=self.roi_size[fidx], aligned=True)
                rois.append(f_roi)
            dis_output = self.discriminator_model(rois)
            return dis_output
        else:
            return None

    def optimizer_step(self, optims: None | list[torch.optim.Optimizer] = None):
        self.scaler.unscale_(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.unscale_(o)
        max_norm = 10.0
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=max_norm)
        if len(self.additional_models) > 0:
            for m in self.additional_models:
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=max_norm * 2)
        self.scaler.step(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.step(o)
        self.scaler.update()
        self.optimizer.zero_grad()
        if optims is not None:
            for o in optims:
                o.zero_grad()
        if self.ema:
            self.ema.update(self.model)

    def _do_train(self, world_size=1):
        if world_size > 1:
            self._setup_ddp(world_size)
        self._setup_train(world_size)
        self.epoch_time = None
        self.epoch_time_start = time.time()
        self.train_time_start = time.time()
        nb = len(self.train_loader)
        nw = max(round(self.args.warmup_epochs * nb), 100) if self.args.warmup_epochs > 0 else -1
        last_opt_step = -1
        self.run_callbacks('on_train_start')
        LOGGER.info(f'Image sizes {self.args.imgsz} train, {self.args.imgsz} val\n'
                    f'Using {self.train_loader.num_workers * (world_size or 1)} dataloader workers\n'
                    f"Logging results to {colorstr('bold', self.save_dir)}\n"
                    f'Starting training for {self.epochs} epochs...')
        if self.args.close_mosaic:
            base_idx = (self.epochs - self.args.close_mosaic) * nb
            self.plot_idx.extend([base_idx, base_idx + 1, base_idx + 2])
        epoch = self.epochs
        self.activate_hook()
        for epoch in range(self.start_epoch, self.epochs):
            self.epoch = epoch
            self.run_callbacks('on_train_epoch_start')
            self.model.train()
            if RANK != -1:
                self.train_loader.sampler.set_epoch(epoch)
            pbar = enumerate(self.train_loader)
            if epoch == (self.epochs - self.args.close_mosaic):
                LOGGER.info('Closing dataloader mosaic')
                if hasattr(self.train_loader.dataset, 'mosaic'):
                    self.train_loader.dataset.mosaic = False
                if hasattr(self.train_loader.dataset, 'close_mosaic'):
                    self.train_loader.dataset.close_mosaic(hyp=self.args)
                self.train_loader.reset()
            if RANK in (-1, 0):
                LOGGER.info(self.progress_string())
                pbar = TQDM(enumerate(self.train_loader), total=nb)
            self.tloss = None
            self.optimizer.zero_grad()
            for i, batch in pbar:
                self.run_callbacks('on_train_batch_start')
                ni = i + nb * epoch
                if ni <= nw:
                    xi = [0, nw]
                    self.accumulate = max(1, np.interp(ni, xi, [1, self.args.nbs / self.batch_size]).round())
                    for j, x in enumerate(self.optimizer.param_groups):
                        x['lr'] = np.interp(
                            ni, xi, [self.args.warmup_bias_lr if j == 0 else 0.0, x['initial_lr'] * self.lf(epoch)])
                        if 'momentum' in x:
                            x['momentum'] = np.interp(ni, xi, [self.args.warmup_momentum, self.args.momentum])
                with torch.cuda.amp.autocast(self.amp):
                    batch = self.preprocess_batch(batch)
                    self.loss, self.loss_items = self.model(batch)
                    if RANK != -1:
                        self.loss *= world_size
                    self.tloss = (self.tloss * i + self.loss_items) / (i + 1) if self.tloss is not None \
                        else self.loss_items
                    source_critics = self.get_dis_output_from_hooked_features(batch)
                    t_batch = self.get_t_batch()
                    t_batch = self.preprocess_batch(t_batch)
                    t_loss, t_loss_item = self.model(t_batch)
                    target_critics = self.get_dis_output_from_hooked_features(t_batch)

                    # Ensure all losses are scalars
                    self_loss_scalar = self.loss if isinstance(self.loss, torch.Tensor) and self.loss.numel() == 1 else self.loss.mean() if isinstance(self.loss, torch.Tensor) else self.loss
                    t_loss_scalar = t_loss if isinstance(t_loss, torch.Tensor) and t_loss.numel() == 1 else t_loss.mean() if isinstance(t_loss, torch.Tensor) else t_loss

                    # Compute adversarial loss with modified threshold and weight
                    if 6 < epoch < self.args.epochs - 50:
                        threshold = 10  # Reduced from 20 to 10
                        loss_d_source = F.relu(torch.ones_like(source_critics) * threshold + source_critics).mean()
                        loss_d_target = F.relu(torch.ones_like(target_critics) * threshold - target_critics).mean()
                        loss_d = loss_d_source + loss_d_target
                    else:
                        loss_d = torch.tensor(0.0, device=self.device)

                    # Combine losses with increased adversarial loss weight
                    self.loss = self_loss_scalar + t_loss_scalar + loss_d * 3  # Increased from 2 to 3

                self.scaler.scale(self.loss).backward()
                if ni - last_opt_step >= self.accumulate:
                    self.optimizer_step(optims=[self.discriminator_model.optim])
                    last_opt_step = ni
                mem = f'{torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0:.3g}G'
                loss_len = self.tloss.shape[0] if len(self.tloss.size()) else 1
                losses = self.tloss if loss_len > 1 else torch.unsqueeze(self.tloss, 0)
                if RANK in (-1, 0):
                    pbar.set_description(
                        ('%11s' * 2 + '%11.4g' * (2 + loss_len)) %
                        (f'{epoch + 1}/{self.epochs}', mem, *losses, batch['cls'].shape[0], batch['img'].shape[-1]))
                    self.run_callbacks('on_batch_end')
                    # Log custom scalars to TensorBoard using the SummaryWriter
                    if self.tensorboard_writer is not None:
                        self.tensorboard_writer.add_scalar('train/critic-source', source_critics.mean().item(), ni)
                        self.tensorboard_writer.add_scalar('train/critic-target', target_critics.mean().item(), ni)
                    if self.args.plots and ni in self.plot_idx:
                        self.plot_training_samples(batch, ni)
                self.run_callbacks('on_train_batch_end')
            self.lr = {f'lr/pg{ir}': x['lr'] for ir, x in enumerate(self.optimizer.param_groups)}
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                self.scheduler.step()
            self.run_callbacks('on_train_epoch_end')
            if RANK in (-1, 0):
                self.ema.update_attr(self.model, include=['yaml', 'nc', 'args', 'names', 'stride', 'class_weights'])
                final_epoch = (epoch + 1 == self.epochs) or self.stopper.possible_stop
                if self.args.val or final_epoch:
                    self.metrics, self.fitness = self.validate()
                self.save_metrics(metrics={**self.label_loss_items(self.tloss), **self.metrics, **self.lr})
                self.stop = self.stopper(epoch + 1, self.fitness)
                if self.args.save or (epoch + 1 == self.epochs):
                    self.deactivate_hook()
                    self.save_model()
                    self.run_callbacks('on_model_save')
                    self.activate_hook()
            tnow = time.time()
            self.epoch_time = tnow - self.epoch_time_start
            self.epoch_time_start = tnow
            self.run_callbacks('on_fit_epoch_end')
            torch.cuda.empty_cache()
            if RANK != -1:
                broadcast_list = [self.stop if RANK == 0 else None]
                dist.broadcast_object_list(broadcast_list, 0)
                if RANK != -1:
                    self.stop = broadcast_list[0]
            if self.stop:
                break
        if RANK in (-1, 0):
            LOGGER.info(f'\n{epoch - self.start_epoch + 1} epochs completed in '
                        f'{(time.time() - self.train_time_start) / 3600:.3f} hours.')
            self.final_eval()
            if self.args.plots:
                self.plot_metrics()
            self.run_callbacks('on_train_end')
        self.deactivate_hook()
        torch.cuda.empty_cache()
        self.run_callbacks('teardown')

def main():
    # Set hyperparameters and random seed
    seed = 95 * 27
    kwargs = {
        'imgsz': 640,
        'epochs': 50,  # Increased from 30 to 50
        'val': True,
        'workers': 2,
        'batch': 16,
        'seed': seed,
        'hsv_s': 0.9,  # Increased from 0.7 to 0.9
        'hsv_v': 0.6,  # Increased from 0.4 to 0.6
        'degrees': 10.0,  # Added rotation (from 0.0 to 10.0)
        'cos_lr': True,  # Enabled cosine learning rate schedule
    }
    seed_everything(seed)

    # Load your pre-trained model
    model = YOLO('yolov8s.yaml').load('/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt')

    # Initialize the custom trainer for domain adaptation
    custom_trainer = partial(CustomTrainer, target_domain_data_cfg='/content/drive/My Drive/pguard/thermal/thermal.yaml')

    # Train with domain adaptation (visible as source, thermal as target)
    model.train(
        custom_trainer,
        data='/content/drive/My Drive/pguard/dataset.yaml',
        name='visible_to_thermal',
        patience=0,
        **kwargs
    )

    # Evaluate the adapted model on the thermal dataset
    model.val(
        data='/content/drive/My Drive/pguard/thermal/thermal.yaml',
        name='val_visible_to_thermal'
    )

if __name__ == '__main__':
    main()

Writing train.py


In [ ]:
# Change to the project directory
%cd /content/Domain-adaptation-object-detection-with-YOLOv8

# Run the training script
!python train.py

In [ ]:
%%writefile train.py
from ultralytics.models import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr, emojis, clean_url
import ultralytics.utils.callbacks.tensorboard as tb_module
import numpy as np
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from functools import partial
import random
import os

def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super(GradientReversalLayer, self).__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h // 2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim_h // 2, dim_h // 2),
                nn.LeakyReLU(0.2, inplace=True),
            ) for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h // 2 * 4, dim_h // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h // 2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        x = x.split(x.shape[1] // 2, dim=1)
        xs = [x[0]]
        for m in self.neck:
            x = m(x[1]) if isinstance(x, tuple) else m(x)
            xs.append(x)
        x = torch.cat(xs, dim=1)
        return self.head(x)

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        if chs is None:
            chs = [64, 128, 256]
            self.chs = chs
            self.f_len = len(chs)
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.p = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i == 0 else chs[i] * 2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64),
                nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(32),
                nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i + 1] if i + 1 < len(chs) else chs[i], kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(chs[i + 1] if i + 1 < len(chs) else chs[i]),
                nn.SiLU(inplace=True),
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, fs: list[torch.tensor]):
        with torch.cuda.amp.autocast(self.amp):
            assert len(fs) == self.f_len, f'Expected {self.f_len} feature maps, got {len(fs)}'
            fs = [self.grl(f) for f in fs]
            x = self.p[0](fs[0])
            for i in range(1, len(fs)):
                x = torch.cat((x, fs[i]), dim=1)
                x = self.p[i](x)
            return self.head(x)

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, *args, **kwargs):
        super(CustomTrainer, self).__init__(*args, **kwargs)
        # Load the target domain dataset (thermal)
        try:
            if target_domain_data_cfg.endswith(('yaml', 'yml')) or self.args.task in ('detect', 'segment', 'pose'):
                self.t_data = check_det_dataset(target_domain_data_cfg)
                # Initialize target datasets directly in __init__
                self.t_trainset, self.t_testset = super().get_dataset()
        except Exception as e:
            raise RuntimeError(emojis(f"Dataset '{clean_url(target_domain_data_cfg)}' error ❌ {e}")) from e
        self.t_iter = None
        self.t_train_loader = None
        self.model_hook_handler = []
        self.model_hook_layer_idx: list[int] = [2, 4, 6]
        self.roi_size = list(reversed([20 * 2 ** i for i in range(len(self.model_hook_layer_idx))]))
        self.model_hooked_features: None | list[torch.tensor] = None
        self.discriminator_model = None
        self.projection_model = None
        self.additional_models = []
        self.tensorboard_writer = None  # Initialize TensorBoard writer
        self.add_callback('on_train_start', self.init_helper_model)
        self.add_callback('on_train_start', self.init_tensorboard_writer)  # Add callback to initialize TensorBoard writer
        self.add_callback('teardown', self.close_tensorboard_writer)  # Add callback to close TensorBoard writer

    def init_helper_model(self, *args, **kwargs):
        self.discriminator_model = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator_model)

    def init_tensorboard_writer(self, *args, **kwargs):
        if RANK in (-1, 0):  # Only initialize on the main process
            self.tensorboard_writer = tb_module.SummaryWriter(log_dir=self.save_dir)  # Use tb_module.SummaryWriter
            LOGGER.info(f"TensorBoard writer initialized at {self.save_dir}")

    def close_tensorboard_writer(self, *args, **kwargs):
        if self.tensorboard_writer is not None:
            self.tensorboard_writer.close()
            LOGGER.info("TensorBoard writer closed")

    def get_t_batch(self):
        if self.t_iter is None:
            self.t_train_loader = self.get_dataloader(self.t_trainset, batch_size=self.batch_size, rank=RANK, mode='train')
            self.t_iter = iter(self.t_train_loader)
        try:
            batch = next(self.t_iter)
        except StopIteration:
            self.t_iter = iter(self.t_train_loader)
            batch = next(self.t_iter)
        return batch

    def activate_hook(self, layer_indices: list[int] = None):
        if layer_indices is not None:
            self.model_hook_layer_idx = layer_indices
        self.model_hooked_features = [None for _ in self.model_hook_layer_idx]
        self.model_hook_handler = \
            [self.model.model[l].register_forward_hook(self.hook_fn(i)) for i, l in enumerate(self.model_hook_layer_idx)]

    def deactivate_hook(self):
        if self.model_hook_handler is not None:
            for hook in self.model_hook_handler:
                hook.remove()
            self.model_hooked_features = None
            self.model_hook_handler = []

    def hook_fn(self, hook_idx):
        def hook(m, i, o):
            self.model_hooked_features[hook_idx] = o
        return hook

    def get_dis_output_from_hooked_features(self, batch):
        if self.model_hooked_features is not None:
            bbox_batch_idx = batch['batch_idx'].unsqueeze(-1)
            bbox = batch['bboxes']
            bbox = box_convert(bbox, 'cxcywh', 'xyxy')
            rois = []
            for fidx, f in enumerate(self.model_hooked_features):
                f_bbox = bbox * f.shape[-1]
                f_bbox = torch.cat((bbox_batch_idx, f_bbox), dim=-1)
                f_roi = roi_align(f, f_bbox.to(f.device), output_size=self.roi_size[fidx], aligned=True)
                rois.append(f_roi)
            dis_output = self.discriminator_model(rois)
            return dis_output
        else:
            return None

    def optimizer_step(self, optims: None | list[torch.optim.Optimizer] = None):
        self.scaler.unscale_(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.unscale_(o)
        max_norm = 10.0
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=max_norm)
        if len(self.additional_models) > 0:
            for m in self.additional_models:
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=max_norm * 2)
        self.scaler.step(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.step(o)
        self.scaler.update()
        self.optimizer.zero_grad()
        if optims is not None:
            for o in optims:
                o.zero_grad()
        if self.ema:
            self.ema.update(self.model)

    def _do_train(self, world_size=1):
        if world_size > 1:
            self._setup_ddp(world_size)
        self._setup_train(world_size)
        self.epoch_time = None
        self.epoch_time_start = time.time()
        self.train_time_start = time.time()
        nb = len(self.train_loader)
        nw = max(round(self.args.warmup_epochs * nb), 100) if self.args.warmup_epochs > 0 else -1
        last_opt_step = -1
        self.run_callbacks('on_train_start')
        LOGGER.info(f'Image sizes {self.args.imgsz} train, {self.args.imgsz} val\n'
                    f'Using {self.train_loader.num_workers * (world_size or 1)} dataloader workers\n'
                    f"Logging results to {colorstr('bold', self.save_dir)}\n"
                    f'Starting training for {self.epochs} epochs...')
        if self.args.close_mosaic:
            base_idx = (self.epochs - self.args.close_mosaic) * nb
            self.plot_idx.extend([base_idx, base_idx + 1, base_idx + 2])
        epoch = self.epochs
        self.activate_hook()
        for epoch in range(self.start_epoch, self.epochs):
            self.epoch = epoch
            self.run_callbacks('on_train_epoch_start')
            self.model.train()
            if RANK != -1:
                self.train_loader.sampler.set_epoch(epoch)
            pbar = enumerate(self.train_loader)
            if epoch == (self.epochs - self.args.close_mosaic):
                LOGGER.info('Closing dataloader mosaic')
                if hasattr(self.train_loader.dataset, 'mosaic'):
                    self.train_loader.dataset.mosaic = False
                if hasattr(self.train_loader.dataset, 'close_mosaic'):
                    self.train_loader.dataset.close_mosaic(hyp=self.args)
                self.train_loader.reset()
            if RANK in (-1, 0):
                LOGGER.info(self.progress_string())
                pbar = TQDM(enumerate(self.train_loader), total=nb)
            self.tloss = None
            self.optimizer.zero_grad()
            for i, batch in pbar:
                self.run_callbacks('on_train_batch_start')
                ni = i + nb * epoch
                if ni <= nw:
                    xi = [0, nw]
                    self.accumulate = max(1, np.interp(ni, xi, [1, self.args.nbs / self.batch_size]).round())
                    for j, x in enumerate(self.optimizer.param_groups):
                        x['lr'] = np.interp(
                            ni, xi, [self.args.warmup_bias_lr if j == 0 else 0.0, x['initial_lr'] * self.lf(epoch)])
                        if 'momentum' in x:
                            x['momentum'] = np.interp(ni, xi, [self.args.warmup_momentum, self.args.momentum])
                with torch.cuda.amp.autocast(self.amp):
                    batch = self.preprocess_batch(batch)
                    self.loss, self.loss_items = self.model(batch)
                    if RANK != -1:
                        self.loss *= world_size
                    self.tloss = (self.tloss * i + self.loss_items) / (i + 1) if self.tloss is not None \
                        else self.loss_items
                    source_critics = self.get_dis_output_from_hooked_features(batch)
                    t_batch = self.get_t_batch()
                    t_batch = self.preprocess_batch(t_batch)
                    t_loss, t_loss_item = self.model(t_batch)
                    target_critics = self.get_dis_output_from_hooked_features(t_batch)

                    # Ensure all losses are scalars
                    self_loss_scalar = self.loss if isinstance(self.loss, torch.Tensor) and self.loss.numel() == 1 else self.loss.mean() if isinstance(self.loss, torch.Tensor) else self.loss
                    t_loss_scalar = t_loss if isinstance(t_loss, torch.Tensor) and t_loss.numel() == 1 else t_loss.mean() if isinstance(t_loss, torch.Tensor) else t_loss

                    # Compute adversarial loss with original threshold and weight
                    if 6 < epoch < self.args.epochs - 50:
                        threshold = 20  # Revert to 20 (previously 10)
                        loss_d_source = F.relu(torch.ones_like(source_critics) * threshold + source_critics).mean()
                        loss_d_target = F.relu(torch.ones_like(target_critics) * threshold - target_critics).mean()
                        loss_d = loss_d_source + loss_d_target
                    else:
                        loss_d = torch.tensor(0.0, device=self.device)

                    # Combine losses with original adversarial loss weight
                    self.loss = self_loss_scalar + t_loss_scalar + loss_d * 2  # Revert to 2 (previously 3)

                self.scaler.scale(self.loss).backward()
                if ni - last_opt_step >= self.accumulate:
                    self.optimizer_step(optims=[self.discriminator_model.optim])
                    last_opt_step = ni
                mem = f'{torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0:.3g}G'
                loss_len = self.tloss.shape[0] if len(self.tloss.size()) else 1
                losses = self.tloss if loss_len > 1 else torch.unsqueeze(self.tloss, 0)
                if RANK in (-1, 0):
                    pbar.set_description(
                        ('%11s' * 2 + '%11.4g' * (2 + loss_len)) %
                        (f'{epoch + 1}/{self.epochs}', mem, *losses, batch['cls'].shape[0], batch['img'].shape[-1]))
                    self.run_callbacks('on_batch_end')
                    # Log custom scalars to TensorBoard using the SummaryWriter
                    if self.tensorboard_writer is not None:
                        self.tensorboard_writer.add_scalar('train/critic-source', source_critics.mean().item(), ni)
                        self.tensorboard_writer.add_scalar('train/critic-target', target_critics.mean().item(), ni)
                    if self.args.plots and ni in self.plot_idx:
                        self.plot_training_samples(batch, ni)
                self.run_callbacks('on_train_batch_end')
            self.lr = {f'lr/pg{ir}': x['lr'] for ir, x in enumerate(self.optimizer.param_groups)}
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                self.scheduler.step()
            self.run_callbacks('on_train_epoch_end')
            if RANK in (-1, 0):
                self.ema.update_attr(self.model, include=['yaml', 'nc', 'args', 'names', 'stride', 'class_weights'])
                final_epoch = (epoch + 1 == self.epochs) or self.stopper.possible_stop
                if self.args.val or final_epoch:
                    self.metrics, self.fitness = self.validate()
                self.save_metrics(metrics={**self.label_loss_items(self.tloss), **self.metrics, **self.lr})
                self.stop = self.stopper(epoch + 1, self.fitness)
                if self.args.save or (epoch + 1 == self.epochs):
                    self.deactivate_hook()
                    self.save_model()
                    self.run_callbacks('on_model_save')
                    self.activate_hook()
            tnow = time.time()
            self.epoch_time = tnow - self.epoch_time_start
            self.epoch_time_start = tnow
            self.run_callbacks('on_fit_epoch_end')
            torch.cuda.empty_cache()
            if RANK != -1:
                broadcast_list = [self.stop if RANK == 0 else None]
                dist.broadcast_object_list(broadcast_list, 0)
                if RANK != -1:
                    self.stop = broadcast_list[0]
            if self.stop:
                break
        if RANK in (-1, 0):
            LOGGER.info(f'\n{epoch - self.start_epoch + 1} epochs completed in '
                        f'{(time.time() - self.train_time_start) / 3600:.3f} hours.')
            self.final_eval()
            if self.args.plots:
                self.plot_metrics()
            self.run_callbacks('on_train_end')
        self.deactivate_hook()
        torch.cuda.empty_cache()
        self.run_callbacks('teardown')

def main():
    # Set hyperparameters and random seed
    seed = 95 * 27
    kwargs = {
        'imgsz': 640,
        'epochs': 50,  # Increased from 30 to 50
        'val': True,
        'workers': 2,
        'batch': 16,
        'seed': seed,
        'hsv_s': 0.9,  # Increased from 0.7 to 0.9
        'hsv_v': 0.6,  # Increased from 0.4 to 0.6
        'degrees': 10.0,  # Added rotation (from 0.0 to 10.0)
        'cos_lr': True,  # Enabled cosine learning rate schedule
    }
    seed_everything(seed)

    # Load your pre-trained model
    model = YOLO('yolov8s.yaml').load('/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt')

    # Initialize the custom trainer for domain adaptation
    custom_trainer = partial(CustomTrainer, target_domain_data_cfg='/content/drive/My Drive/pguard/thermal/thermal.yaml')

    # Train with domain adaptation (visible as source, thermal as target)
    model.train(
        custom_trainer,
        data='/content/drive/My Drive/pguard/dataset.yaml',
        name='visible_to_thermal_reverted',
        patience=0,
        **kwargs
    )

    # Save training results to Google Drive
    import shutil
    shutil.copytree(
        "runs/detect/visible_to_thermal_reverted",
        "/content/drive/My Drive/pguard/runs/adapt/visible_to_thermal_reverted",
        dirs_exist_ok=True
    )
    print("Training results saved to /content/drive/My Drive/pguard/runs/adapt/visible_to_thermal_reverted")

    # Evaluate the adapted model on the thermal dataset
    model.val(
        data='/content/drive/My Drive/pguard/thermal/thermal.yaml',
        name='val_visible_to_thermal',
        augment=False,  # Explicitly disable augmentations during validation
        cache=False  # Disable caching to avoid NumPy compatibility issues
    )

if __name__ == '__main__':
    main()

In [ ]:

%cd /content/Domain-adaptation-object-detection-with-YOLOv8

# Run the training script
!python train.py

In [ ]:
import os

# Check if the weights exist locally
weights_dir = "runs/detect/visible_to_thermal4/weights"
if os.path.exists(weights_dir):
    print(f"Weights directory exists: {weights_dir}")
    print("Files in weights directory:", os.listdir(weights_dir))
else:
    print(f"Weights directory not found: {weights_dir}")

In [ ]:
import shutil

# Copy the training results to Google Drive
shutil.copytree(
    "runs/detect/visible_to_thermal4",
    "/content/drive/My Drive/pguard/runs/adapt/visible_to_thermal_reverted",
    dirs_exist_ok=True
)
print("Training results saved to /content/drive/My Drive/pguard/runs/adapt/visible_to_thermal_reverted")

# Verify the weights are now in Google Drive
gdrive_dir = "/content/drive/My Drive/pguard/runs/adapt/visible_to_thermal_reverted"
weights_dir = os.path.join(gdrive_dir, "weights")
if os.path.exists(weights_dir):
    print(f"Weights found in Google Drive: {os.listdir(weights_dir)}")
else:
    print("Weights directory not found in Google Drive.")

In [ ]:
import shutil

# Save validation results to Google Drive
shutil.copytree(
    "runs/detect/val_visible_to_thermal2",
    "/content/drive/My Drive/pguard/runs/val/visible_to_thermal_reverted2",
    dirs_exist_ok=True
)
print("Validation results saved to /content/drive/My Drive/pguard/runs/val/visible_to_thermal_reverted2")

In [ ]:
import shutil

# Save evaluation metrics to a text file
with open("thermal_evaluation_results_reverted2.txt", "w") as f:
    f.write(f"mAP@0.5: 0.707\n")
    f.write(f"mAP@0.5:0.95: 0.413\n")
    f.write(f"Precision: 0.864\n")
    f.write(f"Recall: 0.501\n")
print("Evaluation metrics saved to thermal_evaluation_results_reverted2.txt")

# Copy the evaluation metrics to Google Drive
shutil.copy(
    "thermal_evaluation_results_reverted2.txt",
    "/content/drive/My Drive/pguard/runs/val/thermal_evaluation_results_reverted2.txt"
)
print("Evaluation metrics saved to /content/drive/My Drive/pguard/runs/val/thermal_evaluation_results_reverted2.txt")

Evaluation metrics saved to thermal_evaluation_results_reverted2.txt
Evaluation metrics saved to /content/drive/My Drive/pguard/runs/val/thermal_evaluation_results_reverted2.txt


In [ ]:
!git clone https://github.com/ultralytics/ultralytics.git
%cd ultralytics
!pip install -e .


fatal: destination path 'ultralytics' already exists and is not an empty directory.
/content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics
Obtaining file:///content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics
ERROR: file:///content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
from ultralytics import YOLO
import os
import random
import shutil

# Load the new model from Google Drive
model = YOLO("/content/drive/My Drive/pguard/runs/adapt/visible_to_thermal_reverted/weights/best.pt")

# Path to thermal images
thermal_images_path = "/content/drive/My Drive/pguard/thermal/images/"
all_images = [os.path.join(thermal_images_path, img) for img in os.listdir(thermal_images_path) if img.endswith(('.jpg', '.jpeg', '.png'))]

# Select the same subset of images (50 images) for consistency
random.seed(42)  # Use the same seed as before for reproducibility
subset_images = random.sample(all_images, min(50, len(all_images)))

# Predict on the subset
results = model.predict(
    source=subset_images,
    save=True,
    device=0,  # Use GPU (A100)
    workers=0,  # Disable DataLoader multiprocessing
    augment=False  # Disable augmentations
)

# Save prediction results to Google Drive
shutil.copytree(
    "runs/detect/predict",
    "/content/drive/My Drive/pguard/runs/detect/thermal_predict_reverted2",
    dirs_exist_ok=True
)
print("Prediction results saved to /content/drive/My Drive/pguard/runs/detect/thermal_predict_reverted2")

In [ ]:
import os

# Paths to images and labels
thermal_images_path = "/content/drive/My Drive/pguard/thermal/images/"
thermal_labels_path = "/content/drive/My Drive/pguard/thermal/labels/"

# List all images and labels
image_files = [f for f in os.listdir(thermal_images_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
label_files = [f for f in os.listdir(thermal_labels_path) if f.endswith('.txt')]

# Check the number of files
print(f"Number of thermal images: {len(image_files)}")
print(f"Number of thermal labels: {len(label_files)}")

# Verify that each image has a corresponding label
matched_pairs = 0
for img in image_files:
    label_name = os.path.splitext(img)[0] + '.txt'
    if label_name in label_files:
        matched_pairs += 1
    else:
        print(f"No label found for image: {img}")

print(f"Number of matched image-label pairs: {matched_pairs}")

In [ ]:
import random
import shutil

# Paths
thermal_images_path = "/content/drive/My Drive/pguard/thermal/val/images/"
thermal_labels_path = "/content/drive/My Drive/pguard/thermal/val/labels/"

# List all images
image_files = [f for f in os.listdir(thermal_images_path) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Select 200 images randomly
num_labeled = 40
random.seed(42)  # For reproducibility
selected_images = random.sample(image_files, num_labeled)

# Create directories for the labeled subset
labeled_images_path = "/content/drive/My Drive/pguard/thermal/thermal_labeled/val/images/"
labeled_labels_path = "/content/drive/My Drive/pguard/thermal/thermal_labeled/val/labels/"
os.makedirs(labeled_images_path, exist_ok=True)
os.makedirs(labeled_labels_path, exist_ok=True)

# Copy the selected images and their labels
for img in selected_images:
    # Copy the image
    src_img = os.path.join(thermal_images_path, img)
    dst_img = os.path.join(labeled_images_path, img)
    shutil.copy(src_img, dst_img)

    # Copy the corresponding label
    label_name = os.path.splitext(img)[0] + '.txt'
    src_label = os.path.join(thermal_labels_path, label_name)
    dst_label = os.path.join(labeled_labels_path, label_name)
    shutil.copy(src_label, dst_label)

print(f"Copied {num_labeled} images and labels to /content/drive/My Drive/pguard/thermal/thermal_labeled/")

Copied 40 images and labels to /content/drive/My Drive/pguard/thermal/thermal_labeled/


In [ ]:
import os
import shutil
import random

# Paths
train_img_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images"
train_label_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels"
val_img_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/val/images"
val_label_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/val/labels"

# Get list of images
images = [f for f in os.listdir(train_img_dir) if f.endswith('.jpg')]
if len(images) != 200:
    print(f"Warning: Found {len(images)} images, expected 200.")
random.shuffle(images)

# Select 40 images for validation
val_images = images[:40]
train_images = images[40:]

# Move validation images and their labels
for img in val_images:
    # Move image
    shutil.move(os.path.join(train_img_dir, img), os.path.join(val_img_dir, img))
    # Move corresponding label
    label_file = img.replace('.jpg', '.txt')
    if os.path.exists(os.path.join(train_label_dir, label_file)):
        shutil.move(os.path.join(train_label_dir, label_file), os.path.join(val_label_dir, label_file))
    else:
        print(f"Warning: Label file {label_file} not found for image {img}")

print(f"Moved {len(val_images)} images to {val_img_dir} and their labels to {val_label_dir}")

Moved 0 images to /content/drive/My Drive/pguard/thermal/thermal_labeled/val/images and their labels to /content/drive/My Drive/pguard/thermal/thermal_labeled/val/labels


In [ ]:
# Write the YAML content to the file
yaml_content = """path: /content/drive/My Drive/pguard/thermal
train: thermal_labeled/train/images
val: thermal_labeled/val/images
nc: 1
names: ["lp"]
"""

with open("/content/drive/My Drive/pguard/thermal/thermal_labeled.yaml", "w") as f:
    f.write(yaml_content)

print("Created thermal_labeled.yaml")





Created thermal_labeled.yaml


In [ ]:
# Write the YAML content to the file
yaml_content = """path: /content/drive/My Drive/pguard/thermal
path: /content/drive/My Drive/pguard/thermal
train: train/images  # 1,226 images
val: val/images      # 526 images
nc: 1
names: ['lp']
"""

with open("/content/drive/My Drive/pguard/thermal/thermal.yaml", "w") as f:
    f.write(yaml_content)

print("Created thermal.yaml")


Created thermal.yaml


In [ ]:
import ultralytics
print(ultralytics.__file__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
/usr/local/lib/python3.11/dist-packages/ultralytics/__init__.py


In [ ]:
!cp /content/Domain-adaptation-object-detection-with-YOLOv8/augment.py /usr/local/lib/python3.11/dist-packages/ultralytics/data/augment.py

cp: cannot stat '/content/Domain-adaptation-object-detection-with-YOLOv8/augment.py': No such file or directory


In [ ]:
# Import necessary libraries
import os
import shutil
from google.colab import files

# Define the file paths
input_file = "/content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment.py"
output_file = "/content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment_modified.py"

# Check if the input file exists
if not os.path.exists(input_file):
    print(f"Error: {input_file} does not exist. Please check the path and ensure the file is available.")
else:
    print(f"Found augment.py at: {input_file}")

    # Define the updated classes and functions as strings
    updated_albumentations = '''
class Albumentations:
    def __init__(self, p=1.0, hsv_v=0.6, is_thermal=False):
        """
        Initialize the Albumentations transform object for YOLO bbox formatted parameters.

        Args:
            p (float): Probability of applying the augmentations.
            hsv_v (float): Brightness/contrast adjustment factor (from training kwargs).
            is_thermal (bool): Flag to indicate if the input is a thermal image (grayscale).
        """
        self.p = p
        self.hsv_v = hsv_v
        self.is_thermal = is_thermal
        self.transform = None
        prefix = colorstr("albumentations: ")

        try:
            import albumentations as A
            check_version(A.__version__, "1.0.3", hard=True)

            # Common transforms for both visible and thermal
            common_transforms = [
                A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),  # Add noise to simulate thermal sensor noise
                A.GaussianBlur(blur_limit=(3, 7), p=0.3),     # Simulate heat diffusion blur
                A.RandomBrightnessContrast(
                    brightness_limit=self.hsv_v, contrast_limit=self.hsv_v, p=0.5
                ),  # Adjust intensity
            ]

            # Visible-specific transforms (RGB images)
            visible_transforms = [
                A.ToGray(p=0.01),  # Occasionally convert to grayscale to help domain adaptation
                A.CLAHE(p=0.01),   # Contrast enhancement for visible images
            ]

            # Thermal-specific transforms (grayscale images)
            thermal_transforms = [
                A.RandomBrightnessContrast(
                    brightness_limit=self.hsv_v * 1.5, contrast_limit=self.hsv_v * 1.5, p=0.7
                ),  # Stronger intensity variation for thermal
                A.ImageCompression(quality_range=(75, 100), p=0.1),  # Simulate thermal image quality loss
            ]

            # Combine transforms based on image type
            T = common_transforms
            if self.is_thermal:
                T.extend(thermal_transforms)
            else:
                T.extend(visible_transforms)

            # Check for spatial transforms
            spatial_transforms = {
                "Affine", "BBoxSafeRandomCrop", "CenterCrop", "CoarseDropout", "Crop", "CropAndPad",
                "CropNonEmptyMaskIfExists", "D4", "ElasticTransform", "Flip", "GridDistortion",
                "GridDropout", "HorizontalFlip", "Lambda", "LongestMaxSize", "MaskDropout", "MixUp",
                "Morphological", "NoOp", "OpticalDistortion", "PadIfNeeded", "Perspective",
                "PiecewiseAffine", "PixelDropout", "RandomCrop", "RandomCropFromBorders",
                "RandomGridShuffle", "RandomResizedCrop", "RandomRotate90", "RandomScale",
                "RandomSizedBBoxSafeCrop", "RandomSizedCrop", "Resize", "Rotate", "SafeRotate",
                "ShiftScaleRotate", "SmallestMaxSize", "Transpose", "VerticalFlip", "XYMasking",
            }

            self.contains_spatial = any(transform.__class__.__name__ in spatial_transforms for transform in T)
            self.transform = (
                A.Compose(T, bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]))
                if self.contains_spatial
                else A.Compose(T)
            )
            if hasattr(self.transform, "set_random_seed"):
                self.transform.set_random_seed(torch.initial_seed())
            LOGGER.info(prefix + ", ".join(f"{x}".replace("always_apply=False, ", "") for x in T if x.p))
        except ImportError:
            LOGGER.warning(f"{prefix}Albumentations not installed, skipping.")
        except Exception as e:
            LOGGER.info(f"{prefix}{e}")

    def __call__(self, labels):
        """
        Apply Albumentations transformations to input labels.

        Args:
            labels (dict): A dictionary containing image data and annotations.
        Returns:
            (dict): The input dictionary with augmented image and updated annotations.
        """
        if self.transform is None or random.random() > self.p:
            return labels

        img = labels["img"]
        # Automatically detect if the image is thermal (grayscale)
        self.is_thermal = len(img.shape) == 2 or img.shape[2] == 1
        if self.is_thermal and len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)  # Convert grayscale to 3-channel for Albumentations

        if self.contains_spatial:
            cls = labels["cls"]
            if len(cls):
                labels["instances"].convert_bbox("xywh")
                labels["instances"].normalize(*img.shape[:2][::-1])
                bboxes = labels["instances"].bboxes
                new = self.transform(image=img, bboxes=bboxes, class_labels=cls)
                if len(new["class_labels"]) > 0:
                    labels["img"] = new["image"]
                    labels["cls"] = np.array(new["class_labels"])
                    bboxes = np.array(new["bboxes"], dtype=np.float32)
                labels["instances"].update(bboxes=bboxes)
        else:
            labels["img"] = self.transform(image=img)["image"]

        # Convert back to grayscale if thermal
        if self.is_thermal:
            labels["img"] = cv2.cvtColor(labels["img"], cv2.COLOR_BGR2GRAY)
        return labels
'''

    updated_random_hsv = '''
class RandomHSV:
    def __init__(self, hgain=0.5, sgain=0.5, vgain=0.5) -> None:
        self.hgain = hgain
        self.sgain = sgain
        self.vgain = vgain

    def __call__(self, labels):
        img = labels["img"]
        is_thermal = len(img.shape) == 2 or img.shape[2] == 1

        if is_thermal and (self.vgain):
            # For thermal images, only adjust value (brightness)
            dtype = img.dtype
            r = np.random.uniform(-1, 1) * self.vgain  # Random gain for value
            img = np.clip(img * (1 + r), 0, 255).astype(dtype)
        elif not is_thermal and (self.hgain or self.sgain or self.vgain):
            # For visible images, apply full HSV augmentation
            dtype = img.dtype
            r = np.random.uniform(-1, 1, 3) * [self.hgain, self.sgain, self.vgain]
            x = np.arange(0, 256, dtype=r.dtype)
            lut_hue = ((x + r[0] * 180) % 180).astype(dtype)
            lut_sat = np.clip(x * (r[1] + 1), 0, 255).astype(dtype)
            lut_val = np.clip(x * (r[2] + 1), 0, 255).astype(dtype)
            lut_sat[0] = 0  # Prevent pure white changing color

            hue, sat, val = cv2.split(cv2.cvtColor(img, cv2.COLOR_BGR2HSV))
            im_hsv = cv2.merge((cv2.LUT(hue, lut_hue), cv2.LUT(sat, lut_sat), cv2.LUT(val, lut_val)))
            cv2.cvtColor(im_hsv, cv2.COLOR_HSV2BGR, dst=img)

        labels["img"] = img
        return labels
'''

    updated_random_perspective = '''
class RandomPerspective:
    def __init__(
        self, degrees=5.0, translate=0.05, scale=0.2, shear=2.0, perspective=0.0, border=(0, 0), pre_transform=None
    ):
        self.degrees = degrees  # Reduced rotation range to ±5 degrees
        self.translate = translate  # Reduced translation to 5% of image size
        self.scale = scale  # Reduced scaling to 0.8–1.2x
        self.shear = shear  # Reduced shear to ±2 degrees
        self.perspective = perspective  # Disabled perspective distortion
        self.border = border
        self.pre_transform = pre_transform
'''

    updated_v8_transforms = '''
def v8_transforms(dataset, imgsz, hyp, stretch=False):
    mosaic = Mosaic(dataset, imgsz=imgsz, p=hyp.mosaic)
    affine = RandomPerspective(
        degrees=hyp.degrees,
        translate=hyp.translate,
        scale=hyp.scale,
        shear=hyp.shear,
        perspective=hyp.perspective,
        pre_transform=None if stretch else LetterBox(new_shape=(imgsz, imgsz)),
    )

    pre_transform = Compose([mosaic, affine])
    if hyp.copy_paste_mode == "flip":
        pre_transform.insert(1, CopyPaste(p=hyp.copy_paste, mode=hyp.copy_paste_mode))
    else:
        pre_transform.append(
            CopyPaste(
                dataset,
                pre_transform=Compose([Mosaic(dataset, imgsz=imgsz, p=hyp.mosaic), affine]),
                p=hyp.copy_paste,
                mode=hyp.copy_paste_mode,
            )
        )
    flip_idx = dataset.data.get("flip_idx", [])
    if dataset.use_keypoints:
        kpt_shape = dataset.data.get("kpt_shape", None)
        if len(flip_idx) == 0 and hyp.fliplr > 0.0:
            hyp.fliplr = 0.0
            LOGGER.warning("WARNING ⚠️ No 'flip_idx' array defined in data.yaml, setting augmentation 'fliplr=0.0'")
        elif flip_idx and (len(flip_idx) != kpt_shape[0]):
            raise ValueError(f"data.yaml flip_idx={flip_idx} length must be equal to kpt_shape[0]={kpt_shape[0]}")

    # Pass hsv_v to Albumentations and dynamically determine is_thermal based on dataset
    return Compose(
        [
            pre_transform,
            MixUp(dataset, pre_transform=pre_transform, p=hyp.mixup),
            Albumentations(p=1.0, hsv_v=hyp.hsv_v),  # Pass hsv_v
            RandomHSV(hgain=hyp.hsv_h, sgain=hyp.hsv_s, vgain=hyp.hsv_v),
            RandomFlip(direction="vertical", p=hyp.flipud),
            RandomFlip(direction="horizontal", p=hyp.fliplr, flip_idx=flip_idx),
        ]
    )
'''

    # Function to replace a class or function definition in the file content
    def replace_class_or_function(content, class_name, updated_code):
        import re
        # Find the start and end of the class/function definition
        pattern = rf"(class\s+{class_name}\s*[\s\S]*?)(?=\nclass|\ndef|\n$)"
        match = re.search(pattern, content)
        if match:
            start, end = match.span()
            content = content[:start] + updated_code + content[end:]
        else:
            # If the class/function doesn't exist, append it
            content += "\n\n" + updated_code
        return content

    # Function to replace a function definition
    def replace_function(content, func_name, updated_code):
        import re
        pattern = rf"(def\s+{func_name}\s*\([\s\S]*?)(?=\nclass|\ndef|\n$)"
        match = re.search(pattern, content)
        if match:
            start, end = match.span()
            content = content[:start] + updated_code + content[end:]
        else:
            content += "\n\n" + updated_code
        return content

    # Create a backup of the original file
    backup_file = input_file + ".backup"
    shutil.copyfile(input_file, backup_file)
    print(f"Created backup: {backup_file}")

    # Read the original file content
    with open(input_file, "r", encoding="utf-8") as f:
        content = f.read()

    # Replace the classes and functions
    content = replace_class_or_function(content, "Albumentations", updated_albumentations)
    content = replace_class_or_function(content, "RandomHSV", updated_random_hsv)
    content = replace_class_or_function(content, "RandomPerspective", updated_random_perspective)
    content = replace_function(content, "v8_transforms", updated_v8_transforms)

    # Write the modified content to a new file
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Modified file saved as: {output_file}")

    # Download the modified file
    files.download(output_file)
    print("Downloading augment_modified.py...")

Found augment.py at: /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment.py
Created backup: /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment.py.backup
Modified file saved as: /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment_modified.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!mv /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment_modified (1).py /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment.py

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `mv /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment_modified (1).py /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment.py'


In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload augment_modified.py
!mv augment_modified.py /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/ultralytics/data/augment.py

Saving augment_modified.py to augment_modified.py


In [ ]:
!pip install opencv-python

In [ ]:
%cd /content/Domain-adaptation-object-detection-with-YOLOv8

!pip install albumentations
!pip install numpy==1.26.4


/content/Domain-adaptation-object-detection-with-YOLOv8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 78.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:

!wget https://raw.githubusercontent.com/ultralytics/ultralytics/v8.3.107/ultralytics/data/augment.py -O /usr/local/lib/python3.11/dist-packages/ultralytics/data/augment.py

--2025-04-17 09:07:50--  https://raw.githubusercontent.com/ultralytics/ultralytics/v8.3.107/ultralytics/data/augment.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 124887 (122K) [text/plain]
Saving to: ‘/usr/local/lib/python3.11/dist-packages/ultralytics/data/augment.py’

/usr/local/lib/pyth 100%[===================>] 121.96K  --.-KB/s    in 0.01s   

2025-04-17 09:07:50 (8.22 MB/s) - ‘/usr/local/lib/python3.11/dist-packages/ultralytics/data/augment.py’ saved [124887/124887]



In [ ]:
!mkdir -p /content/Domain-adaptation-object-detection-with-YOLOv8
%cd /content/Domain-adaptation-object-detection-with-YOLOv8

/content/Domain-adaptation-object-detection-with-YOLOv8


In [ ]:
!pip show ultralytics

Name: ultralytics
Version: 8.3.107
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation and image classification.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.11/dist-packages
Requires: matplotlib, numpy, opencv-python, pandas, pillow, psutil, py-cpuinfo, pyyaml, requests, scipy, seaborn, torch, torchvision, tqdm, ultralytics-thop
Required-by: 


In [ ]:
!cp /usr/local/lib/python3.11/dist-packages/ultralytics/data/augment.py /content/Domain-adaptation-object-detection-with-YOLOv8/ultralytics/augment.py

In [ ]:
!rm -rf /usr/local/lib/python3.11/dist-packages/ultralytics/__pycache__
!rm -rf /usr/local/lib/python3.11/dist-packages/ultralytics/data/__pycache__

In [ ]:
!pip install ultralytics==8.0.0  # Replace with the version you’re using

In [ ]:
import sys
print(sys.executable)  # Path to the Python executable
print(sys.path)  # List of paths where Python looks for modules

/usr/bin/python3
['/content', '/env/python', '/usr/lib/python311.zip', '/usr/lib/python3.11', '/usr/lib/python3.11/lib-dynload', '', '/usr/local/lib/python3.11/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.11/dist-packages/IPython/extensions', '/root/.ipython']


In [ ]:
import torch
torch.cuda.empty_cache()


In [ ]:
!pip install tensorboard

In [ ]:
!pip install timeout-decorator

  Preparing metadata (setup.py) ... done
  Created wheel for timeout-decorator: filename=timeout_decorator-0.5.0-py3-none-any.whl size=5006 sha256=cead1b39429a119bfeba28fc7c4962aadb944e02caa7fc2239548560f1bd61f6
  Stored in directory: /root/.cache/pip/wheels/aa/cd/d1/51736c6b95846b2613a520ce146a8f305c4016a987bc9faec7
Successfully built timeout-decorator


In [ ]:
%%writefile train.py


from ultralytics import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr, emojis, clean_url
import ultralytics.utils.callbacks.tensorboard as tb_module
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from functools import partial
import random
import shutil
import time
import pandas as pd
import timeout_decorator
import os






def seed_everything(seed=9527):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super(GradientReversalLayer, self).__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h // 2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim_h // 2, dim_h // 2),
                nn.LeakyReLU(0.2, inplace=True),
            ) for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h // 2 * 4, dim_h // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h // 2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        x = x.split(x.shape[1] // 2, dim=1)
        xs = [x[0]]
        for m in self.neck:
            x = m(x[1]) if isinstance(x, tuple) else m(x)
            xs.append(x)
        x = torch.cat(xs, dim=1)
        return self.head(x)

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        if chs is None:
            chs = [64, 128, 256]
            self.chs = chs
            self.f_len = len(chs)
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.p = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i == 0 else chs[i] * 2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64),
                nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(32),
                nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i + 1] if i + 1 < len(chs) else chs[i], kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(chs[i + 1] if i + 1 < len(chs) else chs[i]),
                nn.SiLU(inplace=True),
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, fs: list[torch.tensor]):
        with torch.cuda.amp.autocast(self.amp):
            assert len(fs) == self.f_len, f'Expected {self.f_len} feature maps, got {len(fs)}'
            fs = [self.grl(f) for f in fs]
            x = self.p[0](fs[0])
            for i in range(1, len(fs)):
                x = torch.cat((x, fs[i]), dim=1)
                x = self.p[i](x)
            return self.head(x)

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data=None, *args, **kwargs):
        super(CustomTrainer, self).__init__(*args, **kwargs)
        LOGGER.info(f"Initializing CustomTrainer with target_domain_data_cfg: {target_domain_data_cfg}, labeled_thermal_data: {labeled_thermal_data}")
        self.original_data_cfg = self.args.data

        # Load target domain (thermal) dataset
        try:
            if target_domain_data_cfg.endswith(('yaml', 'yml')) or self.args.task in ('detect', 'segment', 'pose'):
                self.t_data = check_det_dataset(target_domain_data_cfg)
                LOGGER.info(f"Target domain data validated with {len(self.t_data.get('train', []))} train images")
                self.args.data = target_domain_data_cfg
                self.t_trainset, self.t_testset = self.get_dataset()
                self.args.data = self.original_data_cfg
                LOGGER.info(f"Target domain dataset loaded with {len(self.t_trainset)} samples")
        except Exception as e:
            raise RuntimeError(emojis(f"Dataset '{clean_url(target_domain_data_cfg)}' error ❌ {e}")) from e
        self.t_iter = None
        self.t_train_loader = None

        # Load labeled thermal dataset for semi-supervised learning
        self.labeled_thermal_data = None
        self.labeled_thermal_iter = None
        self.labeled_thermal_trainset = None
        if labeled_thermal_data:
            self.labeled_thermal_data = check_det_dataset(labeled_thermal_data)
            LOGGER.info(f"Labeled thermal data validated: {len(self.labeled_thermal_data.get('train', []))} train images" if self.labeled_thermal_data else "No labeled thermal data provided")
            train_images = self.labeled_thermal_data.get('train', [])
            if len(train_images) != 200:
                LOGGER.warning(f"Expected 200 labeled thermal images, found {len(train_images)}. Proceeding anyway.")
            try:
                self.args.data = self.labeled_thermal_data['yaml_file']
                self.labeled_thermal_trainset = self.get_dataset()[0]
                self.args.data = self.original_data_cfg
                LOGGER.info(f"Labeled thermal dataset loaded with {len(self.labeled_thermal_trainset)} samples")
            except Exception as e:
                LOGGER.error(f"Failed to initialize labeled_thermal_trainset: {e}")
                self.labeled_thermal_trainset = None

        # Define data augmentation for visible and thermal datasets
        self.visible_augs = {
            'hsv_h': 0.015,
            'hsv_s': 0.9,
            'hsv_v': 0.6,
            'degrees': 10.0,
            'translate': 0.1,
            'scale': 0.5,
            'mosaic': True
        }
        self.thermal_augs = {
            'hsv_h': 0.0,
            'hsv_s': 0.0,
            'hsv_v': 0.2,
            'degrees': 5.0,
            'translate': 0.05,
            'scale': 0.2,
            'mosaic': False
        }

        self.model_hook_handler = []
        self.model_hook_layer_idx: list[int] = [2, 4, 6]
        self.roi_size = list(reversed([20 * 2 ** i for i in range(len(self.model_hook_layer_idx))]))
        self.model_hooked_features: None | list[torch.tensor] = None
        self.discriminator_model = None
        self.additional_models = []
        self.tensorboard_writer = None
        self.thermal_metrics = []
        self.add_callback('on_train_start', self.init_helper_model)
        self.add_callback('on_train_start', self.init_tensorboard_writer)
        self.add_callback('teardown', self.close_tensorboard_writer)

    def init_helper_model(self, *args, **kwargs):
        LOGGER.info("Initializing helper model")
        self.discriminator_model = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator_model)

    def init_tensorboard_writer(self, *args, **kwargs):
        if RANK in (-1, 0):
            LOGGER.info(f"Initializing TensorBoard writer at {self.save_dir}")
            self.tensorboard_writer = tb_module.SummaryWriter(log_dir=self.save_dir)

    def close_tensorboard_writer(self, *args, **kwargs):
        if self.tensorboard_writer is not None:
            LOGGER.info("Closing TensorBoard writer")
            self.tensorboard_writer.close()

    def save_thermal_metrics_to_csv(self):
        if self.thermal_metrics:
            df = pd.DataFrame(self.thermal_metrics)
            csv_path = os.path.join(self.save_dir, 'thermal_results.csv')
            df.to_csv(csv_path, index=False)
            LOGGER.info(f"Thermal metrics saved to {csv_path}")

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            if dataset == self.trainset:
                augs = self.visible_augs
            elif dataset == self.t_trainset:
                augs = self.thermal_augs
            elif dataset == self.labeled_thermal_trainset:
                augs = self.thermal_augs
            else:
                augs = {}

            if hasattr(dataset, 'hsv_h'):
                dataset.hsv_h = augs.get('hsv_h', 0.0)
                dataset.hsv_s = augs.get('hsv_s', 0.0)
                dataset.hsv_v = augs.get('hsv_v', 0.0)
                dataset.degrees = augs.get('degrees', 0.0)
                dataset.translate = augs.get('translate', 0.0)
                dataset.scale = augs.get('scale', 0.0)
                dataset.mosaic = augs.get('mosaic', True)

        return super().get_dataloader(dataset, batch_size, rank, mode)

    def get_t_batch(self):
        if self.t_iter is None:
            LOGGER.info("Initializing target domain data iterator")
            self.t_train_loader = self.get_dataloader(self.t_trainset, batch_size=self.batch_size, rank=RANK, mode='train')
            self.t_iter = iter(self.t_train_loader)
        try:
            batch = next(self.t_iter)
        except StopIteration:
            LOGGER.info("Resetting target domain data iterator")
            self.t_iter = iter(self.t_train_loader)
            batch = next(self.t_iter)
        return batch

    def get_labeled_thermal_batch(self):
        if self.labeled_thermal_data is None or self.labeled_thermal_trainset is None:
            return None
        if self.labeled_thermal_iter is None:
            LOGGER.info("Initializing labeled thermal data iterator")
            self.labeled_thermal_loader = self.get_dataloader(self.labeled_thermal_trainset, batch_size=self.batch_size, rank=RANK, mode='train')
            self.labeled_thermal_iter = iter(self.labeled_thermal_loader)
        try:
            batch = next(self.labeled_thermal_iter)
        except StopIteration:
            LOGGER.info("Resetting labeled thermal data iterator")
            self.labeled_thermal_iter = iter(self.labeled_thermal_loader)
            batch = next(self.labeled_thermal_iter)
        return batch

    def activate_hook(self, layer_indices: list[int] = None):
        if layer_indices is not None:
            self.model_hook_layer_idx = layer_indices
        self.model_hooked_features = [None for _ in self.model_hook_layer_idx]
        self.model_hook_handler = \
            [self.model.model[l].register_forward_hook(self.hook_fn(i)) for i, l in enumerate(self.model_hook_layer_idx)]

    def deactivate_hook(self):
        if self.model_hook_handler is not None:
            for hook in self.model_hook_handler:
                hook.remove()
            self.model_hooked_features = None
            self.model_hook_handler = []

    def hook_fn(self, hook_idx):
        def hook(m, i, o):
            self.model_hooked_features[hook_idx] = o
        return hook

    def get_dis_output_from_hooked_features(self, batch):
        if self.model_hooked_features is not None:
            bbox_batch_idx = batch['batch_idx'].unsqueeze(-1)
            bbox = batch['bboxes']
            bbox = box_convert(bbox, 'cxcywh', 'xyxy')
            rois = []
            for fidx, f in enumerate(self.model_hooked_features):
                f_bbox = bbox * f.shape[-1]
                f_bbox = torch.cat((bbox_batch_idx, f_bbox), dim=-1)
                f_roi = roi_align(f, f_bbox.to(f.device), output_size=self.roi_size[fidx], aligned=True)
                rois.append(f_roi)
            dis_output = self.discriminator_model(rois)
            return dis_output
        else:
            return None

    def optimizer_step(self, optims: None | list[torch.optim.Optimizer] = None):
        self.scaler.unscale_(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.unscale_(o)
        max_norm = 10.0
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=max_norm)
        if len(self.additional_models) > 0:
            for m in self.additional_models:
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=max_norm * 2)
        self.scaler.step(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.step(o)
        self.scaler.update()
        self.optimizer.zero_grad()
        if optims is not None:
            for o in optims:
                o.zero_grad()
        if self.ema:
            self.ema.update(self.model)

    def _do_train(self, world_size=1):
        if world_size > 1:
            self._setup_ddp(world_size)
        self._setup_train(world_size)

        self.epoch_time = None
        self.epoch_time_start = time.time()
        self.train_time_start = time.time()
        nb = len(self.train_loader)
        nw = max(round(self.args.warmup_epochs * nb), 100) if self.args.warmup_epochs > 0 else -1
        last_opt_step = -1
        self.run_callbacks('on_train_start')
        LOGGER.info(f'Image sizes {self.args.imgsz} train, {self.args.imgsz} val\n'
                    f'Using {self.train_loader.num_workers * (world_size or 1)} dataloader workers\n'
                    f"Logging results to {colorstr('bold', self.save_dir)}\n"
                    f'Starting training for {self.epochs} epochs...')
        if self.args.close_mosaic:
            base_idx = (self.epochs - self.args.close_mosaic) * nb
            self.plot_idx.extend([base_idx, base_idx + 1, base_idx + 2])
        self.activate_hook()

        save_dir_drive = "/content/drive/My Drive/pguard/checkpoints/visible_to_thermal_semisupervised"
        os.makedirs(save_dir_drive, exist_ok=True)

        for epoch in range(self.start_epoch, self.epochs):
            LOGGER.info(f"Starting epoch {epoch + 1}")
            self.epoch = epoch
            self.run_callbacks('on_train_epoch_start')
            self.model.train()
            if RANK != -1:
                self.train_loader.sampler.set_epoch(epoch)
            pbar = enumerate(self.train_loader)
            if epoch == (self.epochs - self.args.close_mosaic):
                LOGGER.info('Closing dataloader mosaic')
                if hasattr(self.train_loader.dataset, 'mosaic'):
                    self.train_loader.dataset.mosaic = False
                if hasattr(self.train_loader.dataset, 'close_mosaic'):
                    self.train_loader.dataset.close_mosaic(hyp=self.args)
                self.train_loader.reset()
            if RANK in (-1, 0):
                LOGGER.info(self.progress_string())
                pbar = CustomTQDM(enumerate(self.train_loader), total=nb)
            self.tloss = None
            self.optimizer.zero_grad()
            for i, batch in pbar:
                self.run_callbacks('on_train_batch_start')
                ni = i + nb * epoch
                if ni <= nw:
                    xi = [0, nw]
                    self.accumulate = max(1, np.interp(ni, xi, [1, self.args.nbs / self.batch_size]).round())
                    for j, x in enumerate(self.optimizer.param_groups):
                        x['lr'] = np.interp(
                            ni, xi, [self.args.warmup_bias_lr if j == 0 else 0.0, x['initial_lr'] * self.lf(epoch)])
                        if 'momentum' in x:
                            x['momentum'] = np.interp(ni, xi, [self.args.warmup_momentum, self.args.momentum])
                try:
                    with torch.cuda.amp.autocast(self.amp):
                        # Source (visible) domain forward pass
                        batch = self.preprocess_batch(batch)
                        self.loss, self.loss_items = self.model(batch)
                        if RANK != -1:
                            self.loss *= world_size
                        self.tloss = (self.tloss * i + self.loss_items) / (i + 1) if self.tloss is not None \
                            else self.loss_items
                        source_critics = self.get_dis_output_from_hooked_features(batch)

                        # Target (thermal) domain forward pass
                        t_batch = self.get_t_batch()
                        t_batch = self.preprocess_batch(t_batch)
                        target_critics = self.get_dis_output_from_hooked_features(t_batch)

                        self_loss_scalar = self.loss if isinstance(self.loss, torch.Tensor) and self.loss.numel() == 1 else self.loss.mean() if isinstance(self.loss, torch.Tensor) else self.loss

                        # Domain adaptation loss
                        if 6 < epoch < self.args.epochs - 50:
                            threshold = 10  # Adjusted threshold from 20 to 10 as in the second script
                            loss_d_source = F.relu(torch.ones_like(source_critics) * threshold + source_critics).mean()
                            loss_d_target = F.relu(torch.ones_like(target_critics) * threshold - target_critics).mean()
                            loss_d = loss_d_source + loss_d_target
                        else:
                            loss_d = torch.tensor(0.0, device=self.device)

                        # Semi-supervised learning with labeled thermal data
                        labeled_loss = 0.0
                        if self.labeled_thermal_data and self.labeled_thermal_trainset is not None:
                            labeled_batch = self.get_labeled_thermal_batch()
                            if labeled_batch:
                                labeled_batch = self.preprocess_batch(labeled_batch)
                                labeled_loss, _ = self.model(labeled_batch)
                                labeled_samples = len(self.labeled_thermal_loader) * self.batch_size if self.labeled_thermal_loader else 200
                                weight = min(1.0, (epoch + 1) / 10) * (nb * self.batch_size / labeled_samples) / 10
                                weight = min(weight, 0.5)
                                labeled_loss = labeled_loss.mean() * weight

                        # Combine losses
                        self.loss = self_loss_scalar + loss_d * 3 + labeled_loss

                        # TensorBoard logging
                        if RANK in (-1, 0) and self.tensorboard_writer is not None:
                            self.tensorboard_writer.add_scalar('train/self_loss', self_loss_scalar.item(), ni)
                            self.tensorboard_writer.add_scalar('train/loss_d', loss_d.item(), ni)
                            self.tensorboard_writer.add_scalar('train/labeled_loss', labeled_loss.item(), ni)
                            self.tensorboard_writer.add_scalar('train/labeled_weight', weight, ni)
                            self.tensorboard_writer.add_scalar('train/critic-source', source_critics.mean().item(), ni)
                            self.tensorboard_writer.add_scalar('train/critic-target', target_critics.mean().item(), ni)
                            mem_allocated = torch.cuda.memory_allocated() / 1E9 if torch.cuda.is_available() else 0
                            mem_reserved = torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0
                            self.tensorboard_writer.add_scalar('train/mem_allocated', mem_allocated, ni)
                            self.tensorboard_writer.add_scalar('train/mem_reserved', mem_reserved, ni)

                    self.scaler.scale(self.loss).backward()
                    if ni - last_opt_step >= self.accumulate:
                        self.optimizer_step(optims=[self.discriminator_model.optim])
                        last_opt_step = ni
                except Exception as e:
                    LOGGER.error(f"Error during training batch {i}: {e}")
                    continue

                mem = f'{torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0:.3g}G'
                loss_len = self.tloss.shape[0] if len(self.tloss.size()) else 1
                losses = self.tloss if loss_len > 1 else torch.unsqueeze(self.tloss, 0)
                if RANK in (-1, 0):
                    mem_allocated = torch.cuda.memory_allocated() / 1E9 if torch.cuda.is_available() else 0
                    mem_reserved = torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0
                    pbar.set_description(
                        ('%11s' * 2 + '%11.4g' * (2 + loss_len) + ' Mem: %.3gG/%.3gG') %
                        (f'{epoch + 1}/{self.epochs}', mem, *losses, batch['cls'].shape[0], batch['img'].shape[-1],
                         mem_allocated, mem_reserved))
                    self.run_callbacks('on_batch_end')
                    if self.args.plots and ni in self.plot_idx:
                        self.plot_training_samples(batch, ni)
                self.run_callbacks('on_train_batch_end')

            self.lr = {f'lr/pg{ir}': x['lr'] for ir, x in enumerate(self.optimizer.param_groups)}
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                self.scheduler.step()
            self.run_callbacks('on_train_epoch_end')
            if RANK in (-1, 0):
                self.ema.update_attr(self.model, include=['yaml', 'nc', 'args', 'names', 'stride', 'class_weights'])
                final_epoch = (epoch + 1 == self.epochs) or self.stopper.possible_stop
                if self.args.val or final_epoch:
                    self.metrics, self.fitness = self.validate()
                self.save_metrics(metrics={**self.label_loss_items(self.tloss), **self.metrics, **self.lr})
                self.stop = self.stopper(epoch + 1, self.fitness)
                if self.args.save or (epoch + 1 == self.epochs):
                    self.deactivate_hook()
                    self.save_model()
                    source_dir = os.path.join(str(self.save_dir), "weights")
                    for checkpoint in ["last.pt", "best.pt"]:
                        source_path = os.path.join(source_dir, checkpoint)
                        dest_path = os.path.join(save_dir_drive, f"{checkpoint.replace('.pt', '')}_epoch_{epoch + 1}.pt")
                        if os.path.exists(source_path):
                            shutil.copyfile(source_path, dest_path)
                            LOGGER.info(f"Saved {checkpoint} for epoch {epoch + 1} to {dest_path}")
                    self.run_callbacks('on_model_save')
                    self.activate_hook()

                # Thermal dataset evaluation after each epoch
                LOGGER.info(f"\n=== Thermal Dataset Evaluation After Epoch {epoch + 1} ===")
                LOGGER.info(f"Evaluating on thermal dataset: {self.t_data['yaml_file']}")
                LOGGER.info("This evaluation is separate from the visible dataset training.")
                thermal_eval_model = YOLO(os.path.join(str(self.save_dir), "weights", "last.pt"))
                thermal_metrics = {'epoch': epoch + 1, 'precision': 0.0, 'recall': 0.0, 'map50': 0.0, 'map50_95': 0.0}
                try:
                    thermal_results = thermal_eval_model.val(
                        data=self.t_data['yaml_file'],
                        split="val",
                        augment=False,
                        cache=False,
                        verbose=True
                    )
                    if thermal_results and len(thermal_results.box.p) > 0:
                        thermal_metrics = {
                            'epoch': epoch + 1,
                            'precision': float(thermal_results.box.p[0]),
                            'recall': float(thermal_results.box.r[0]),
                            'map50': float(thermal_results.box.map50),
                            'map50_95': float(thermal_results.box.map)
                        }
                        self.thermal_metrics.append(thermal_metrics)
                        if self.tensorboard_writer:
                            self.tensorboard_writer.add_scalar('thermal/precision', thermal_metrics['precision'], epoch)
                            self.tensorboard_writer.add_scalar('thermal/recall', thermal_metrics['recall'], epoch)
                            self.tensorboard_writer.add_scalar('thermal/map50', thermal_metrics['map50'], epoch)
                            self.tensorboard_writer.add_scalar('thermal/map50_95', thermal_metrics['map50_95'], epoch)
                    else:
                        LOGGER.warning("No valid metrics computed for thermal dataset. Check dataset labels in val split.")
                except Exception as e:
                    LOGGER.warning(f"Failed to evaluate on thermal dataset: {e}")
                LOGGER.info(f"\nThermal Evaluation Results After Epoch {epoch + 1}:")
                LOGGER.info(f"Precision: {thermal_metrics['precision']:.3f}")
                LOGGER.info(f"Recall: {thermal_metrics['recall']:.3f}")
                LOGGER.info(f"mAP@50: {thermal_metrics['map50']:.3f}")
                LOGGER.info(f"mAP@50:95: {thermal_metrics['map50_95']:.3f}")
                LOGGER.info(f"=== Thermal Evaluation Complete for Epoch {epoch + 1} ===\n")

            tnow = time.time()
            self.epoch_time = tnow - self.epoch_time_start
            self.epoch_time_start = tnow
            self.run_callbacks('on_fit_epoch_end')
            torch.cuda.empty_cache()
            if RANK != -1:
                broadcast_list = [self.stop if RANK == 0 else None]
                dist.broadcast_object_list(broadcast_list, 0)
                if RANK != -1:
                    self.stop = broadcast_list[0]
            if self.stop:
                break

        if RANK in (-1, 0):
            LOGGER.info(f'\n{epoch - self.start_epoch + 1} epochs completed in '
                        f'{(time.time() - self.train_time_start) / 3600:.3f} hours.')
            self.final_eval()
            if self.args.plots:
                self.plot_metrics()
            if self.thermal_metrics:
                LOGGER.info("\n=== Average Thermal Dataset Evaluation Metrics Across All Epochs ===")
                avg_precision = np.mean([m['precision'] for m in self.thermal_metrics])
                avg_recall = np.mean([m['recall'] for m in self.thermal_metrics])
                avg_map50 = np.mean([m['map50'] for m in self.thermal_metrics])
                avg_map50_95 = np.mean([m['map50_95'] for m in self.thermal_metrics])
                LOGGER.info(f"Average Precision: {avg_precision:.3f}")
                LOGGER.info(f"Average Recall: {avg_recall:.3f}")
                LOGGER.info(f"Average mAP@50: {avg_map50:.3f}")
                LOGGER.info(f"Average mAP@50:95: {avg_map50_95:.3f}")
                LOGGER.info("=== Average Thermal Evaluation Complete ===\n")
            self.save_thermal_metrics_to_csv()
            self.run_callbacks('on_train_end')
        self.deactivate_hook()
        torch.cuda.empty_cache()
        self.run_callbacks('teardown')

def main():
    seed = 95 * 27
    kwargs = {
        'imgsz': 640,
        'epochs': 30,
        'val': True,
        'workers': 8,
        'batch': 8,
        'seed': seed,
        'hsv_s': 0.9,
        'hsv_v': 0.6,
        'degrees': 10.0,
        'cos_lr': True,
        'patience': 10
    }
    seed_everything(seed)

    # Define dataset paths
    visible_data_path = '/content/drive/My Drive/pguard/dataset.yaml'
    thermal_data_path = '/content/drive/My Drive/pguard/thermal/thermal.yaml'
    labeled_thermal_data_path = '/content/drive/My Drive/pguard/thermal/thermal_labeled.yaml'

    # Validate datasets
    LOGGER.info("Validating datasets...")
    for yaml_path in [visible_data_path, thermal_data_path, labeled_thermal_data_path]:
        try:
            data_info = check_det_dataset(yaml_path)
            LOGGER.info(f"Dataset {yaml_path} validated successfully: {data_info}")
            cache_path = data_info.get('train', '').replace('train', 'labels.cache') if 'train' in data_info else data_info.get('labels', '').replace('labels', 'labels.cache')
            if cache_path and os.path.exists(cache_path):
                LOGGER.info(f"Cache file found: {cache_path}")
            else:
                LOGGER.warning(f"Cache file not found for {yaml_path}: {cache_path}. Forcing re-scan...")
                temp_cache_path = cache_path + '.temp'
                if os.path.exists(cache_path):
                    os.rename(cache_path, temp_cache_path)
                check_det_dataset(yaml_path)
                if os.path.exists(temp_cache_path):
                    os.remove(temp_cache_path)
        except Exception as e:
            LOGGER.error(f"Dataset validation failed for {yaml_path}: {e}")
            raise

    LOGGER.info("Dataset validation completed. Starting model loading...")

    # Load pre-trained weights (default: semi-supervised model)
    drive_weights_path = '/content/drive/My Drive/pguard/runs/detect/visible_to_thermal_semisupervised8/weights/best.pt'
    # Uncomment the line below to use the visible-only trained model instead
    # drive_weights_path = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    local_weights_path = '/content/best.pt'
    if os.path.exists(drive_weights_path):
        LOGGER.info(f"Copying weights from {drive_weights_path} to {local_weights_path}...")
        start_copy_time = time.time()
        shutil.copyfile(drive_weights_path, local_weights_path)
        copy_time = time.time() - start_copy_time
        LOGGER.info(f"Copied weights to {local_weights_path} in {copy_time:.2f} seconds")
        file_size = os.path.getsize(local_weights_path) / (1024 * 1024)
        LOGGER.info(f"Weights file size: {file_size:.2f} MB")
        if file_size < 1:
            LOGGER.error("Weights file appears to be corrupted (size too small).")
            raise RuntimeError("Weights file appears to be corrupted.")
    else:
        LOGGER.error(f"Weights file not found at {drive_weights_path}. Cannot proceed without pre-trained weights.")
        raise FileNotFoundError(f"Weights file not found at {drive_weights_path}")

    LOGGER.info("Clearing GPU memory before model loading...")
    torch.cuda.empty_cache()

    @timeout_decorator.timeout(300)
    def load_model_with_timeout():
        start_time = time.time()
        LOGGER.info(f"Attempting to load weights from {local_weights_path}")
        model = YOLO('yolov8s.yaml')
        LOGGER.info("YOLOv8s model initialized. Loading weights...")
        model.load(local_weights_path)
        elapsed_time = time.time() - start_time
        LOGGER.info(f"Model loading completed in {elapsed_time:.2f} seconds")
        return model

    try:
        model = load_model_with_timeout()
    except TimeoutError:
        LOGGER.error("Model loading timed out after 5 minutes. Aborting.")
        raise TimeoutError("Model loading timed out after 5 minutes. Please check the weights file and runtime resources.")
    except Exception as e:
        LOGGER.error(f"Failed to load model: {e}")
        raise

    # Train with visible-to-thermal domain adaptation and semi-supervised learning
    custom_trainer = partial(
        CustomTrainer,
        target_domain_data_cfg=thermal_data_path,
        labeled_thermal_data=labeled_thermal_data_path
    )
    LOGGER.info("Starting visible-to-thermal training with semi-supervised learning and adversarial domain adaptation...")
    results = model.train(
        custom_trainer,
        data=visible_data_path,
        name='visible_to_thermal_semisupervised_adversarial',
        **kwargs
    )

    # Save training results
    train_dir = str(results.save_dir)
    save_dir_drive = "/content/drive/My Drive/pguard/runs/adapt/visible_to_thermal_semisupervised_adversarial"
    os.makedirs(save_dir_drive, exist_ok=True)
    shutil.copytree(train_dir, save_dir_drive, dirs_exist_ok=True)
    LOGGER.info(f"Training results saved to {save_dir_drive}")

    # Final evaluation on thermal domain
    model.val(data=thermal_data_path, name='val_visible_to_thermal')

if __name__ == '__main__':
    main()

Overwriting train.py


In [ ]:
%cd /content/Domain-adaptation-object-detection-with-YOLOv8
!python train.py

/content/Domain-adaptation-object-detection-with-YOLOv8
E0000 00:00:1744881585.594196    7654 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744881585.599927    7654 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Validating datasets...
100% 755k/755k [00:00<00:00, 20.1MB/s]
Dataset /content/drive/My Drive/pguard/dataset.yaml validated successfully: {'path': PosixPath('/content/drive/My Drive/pguard'), 'train': '/content/drive/My Drive/pguard/train/images', 'val': '/content/drive/My Drive/pguard/val/images', 'test': '/content/drive/My Drive/pguard/test/images', 'nc': 1, 'names': {0: 'lp'}, 'yaml_file': '/content/drive/My Drive/pguard/dataset.yaml'}
Cache file not found for /content/drive/My Drive/pguard/dataset.yaml: /content/drive/My Drive/pguard/labels.cache/images. Forcing re-scan...
Dataset

In [ ]:
import os

def check_image_label_match(image_dir, label_dir):
    """
    Check if images and labels in the specified directories match.

    Args:
        image_dir (str): Path to the directory containing images (e.g., *.jpg, *.jpeg, *.png).
        label_dir (str): Path to the directory containing label files (e.g., *.txt).

    Returns:
        dict: Summary containing:
            - total_images: Number of image files found.
            - total_labels: Number of label files found.
            - matched_pairs: Number of image-label pairs.
            - images_without_labels: List of image filenames without corresponding labels.
            - labels_without_images: List of label filenames without corresponding images.
            - matched_files: List of matched image filenames.
    """
    # Supported image and label extensions
    image_extensions = {'.jpg', '.jpeg', '.png'}
    label_extension = '.txt'

    # Get lists of image and label files
    images = [f for f in os.listdir(image_dir) if os.path.splitext(f)[1].lower() in image_extensions]
    labels = [f for f in os.listdir(label_dir) if f.endswith(label_extension)]

    # Extract base filenames (without extensions)
    image_bases = {os.path.splitext(f)[0] for f in images}
    label_bases = {os.path.splitext(f)[0] for f in labels}

    # Find matches and mismatches
    matched_bases = image_bases & label_bases  # Intersection
    images_without_labels = image_bases - label_bases  # Images missing labels
    labels_without_images = label_bases - image_bases  # Labels missing images

    # Convert bases back to full filenames
    matched_files = [f"{base}{os.path.splitext(f)[1]}" for base in matched_bases for f in images if os.path.splitext(f)[0] == base]
    images_without_labels_files = [f"{base}{os.path.splitext(f)[1]}" for base in images_without_labels for f in images if os.path.splitext(f)[0] == base]
    labels_without_images_files = [f"{base}{label_extension}" for base in labels_without_images]

    # Compile summary
    summary = {
        'total_images': len(images),
        'total_labels': len(labels),
        'matched_pairs': len(matched_bases),
        'images_without_labels': sorted(images_without_labels_files),
        'labels_without_images': sorted(labels_without_images_files),
        'matched_files': sorted(matched_files)
    }

    # Print summary
    print(f"Image-Label Match Check:")
    print(f"  Total Images: {summary['total_images']}")
    print(f"  Total Labels: {summary['total_labels']}")
    print(f"  Matched Pairs: {summary['matched_pairs']}")
    if summary['images_without_labels']:
        print(f"  Images without Labels ({len(summary['images_without_labels'])}):")
        for img in summary['images_without_labels'][:10]:  # Limit to first 10 for brevity
            print(f"    {img}")
        if len(summary['images_without_labels']) > 10:
            print(f"    ... and {len(summary['images_without_labels']) - 10} more")
    else:
        print("  No images without labels.")
    if summary['labels_without_images']:
        print(f"  Labels without Images ({len(summary['labels_without_images'])}):")
        for lbl in summary['labels_without_images'][:10]:
            print(f"    {lbl}")
        if len(summary['labels_without_images']) > 10:
            print(f"    ... and {len(summary['labels_without_images']) - 10} more")
    else:
        print("  No labels without images.")

    return summary

# Example usage for thermal_labeled dataset
image_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images"
label_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels"
result = check_image_label_match(image_dir, label_dir)

Image-Label Match Check:
  Total Images: 200
  Total Labels: 200
  Matched Pairs: 200
  No images without labels.
  No labels without images.


In [ ]:
import os
label_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels"
for label_file in os.listdir(label_dir)[:5]:  # Check first 5 files
    with open(os.path.join(label_dir, label_file), 'r') as f:
        content = f.read().strip()
        print(f"{label_file}: {content if content else 'Empty'}")

TH_1681.txt: 0 0.806641 0.840278 0.300781 0.093750
TH_435.txt: 0 0.762370 0.638021 0.319010 0.092014
TH_419.txt: 0 0.283854 0.481771 0.195312 0.071181
TH_473.txt: 0 0.436849 0.860243 0.316406 0.092014
TH_123.txt: 0 0.783203 0.725694 0.313802 0.093750


In [ ]:
import os

def validate_label_files(label_dir):
    """
    Validate all label files in the directory.
    Returns a list of invalid label files (malformed or incorrect format).
    """
    invalid_files = []
    for label_file in os.listdir(label_dir):
        if not label_file.endswith('.txt'):
            continue
        label_path = os.path.join(label_dir, label_file)
        try:
            with open(label_path, 'r') as f:
                content = f.read().strip().splitlines()
            if not content:  # Empty file is valid
                continue
            for line in content:
                parts = line.split()
                # Expect 5 values: class_id, x_center, y_center, width, height
                if len(parts) != 5:
                    invalid_files.append((label_file, "Incorrect number of values"))
                    break
                # Check class_id is an integer and coordinates are numeric
                if not parts[0].isdigit() or not all(part.replace('.', '', 1).isdigit() for part in parts[1:]):
                    invalid_files.append((label_file, "Non-numeric values"))
                    break
                # Check coordinates are in [0, 1]
                coords = [float(x) for x in parts[1:]]
                if not all(0 <= x <= 1 for x in coords):
                    invalid_files.append((label_file, "Coordinates out of range"))
                    break
        except Exception as e:
            invalid_files.append((label_file, f"Error reading file: {e}"))

    if invalid_files:
        print(f"Found {len(invalid_files)} invalid label files:")
        for file, reason in invalid_files[:10]:  # Limit to first 10
            print(f"  {file}: {reason}")
        if len(invalid_files) > 10:
            print(f"  ... and {len(invalid_files) - 10} more")
    else:
        print("All label files are valid.")
    return invalid_files

label_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels"
invalid_labels = validate_label_files(label_dir)

All label files are valid.


In [ ]:
!cat "/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml"


path: /content/drive/My Drive/pguard/thermal/thermal_labeled
train: /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images
val: /content/drive/My Drive/pguard/thermal/thermal_labeled/val/images
nc: 1
names: ['lp']


In [ ]:
cache_path = "/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.cache"
if os.path.exists(cache_path):
    os.remove(cache_path)
    print("Cleared cache file:", cache_path)
else:
    print("No cache file found.")

No cache file found.


In [ ]:
from PIL import Image
import os

image_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images"
invalid_images = []
for img_file in os.listdir(image_dir):
    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
        try:
            img = Image.open(os.path.join(image_dir, img_file))
            img.verify()  # Check for corruption
            img.close()
        except Exception as e:
            invalid_images.append((img_file, str(e)))

if invalid_images:
    print(f"Found {len(invalid_images)} invalid images:")
    for img, error in invalid_images:
        print(f"  {img}: {error}")
else:
    print("All images are valid.")

All images are valid.


In [ ]:
from ultralytics.data.utils import check_det_dataset
import os

def diagnose_dataset(yaml_path):
    print(f"Diagnosing dataset: {yaml_path}")
    try:
        data = check_det_dataset(yaml_path)
        train_images = data['train']
        print(f"Loaded {len(train_images)} train images")

        # Check directory contents
        image_dir = os.path.join(os.path.dirname(yaml_path), 'train/images')
        if os.path.isdir(image_dir):
            all_images = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            print(f"Directory {image_dir} contains {len(all_images)} images")

            # Find skipped images
            loaded_bases = {os.path.splitext(os.path.basename(img))[0] for img in train_images}
            all_bases = {os.path.splitext(img)[0] for img in all_images}
            skipped_images = all_bases - loaded_bases
            if skipped_images:
                print(f"Skipped images ({len(skipped_images)}):")
                for base in list(skipped_images)[:10]:  # First 10
                    print(f"  {base}")
                if len(skipped_images) > 10:
                    print(f"  ... and {len(skipped_images) - 10} more")
            else:
                print("No images skipped.")

        return data
    except Exception as e:
        print(f"Error during diagnosis: {e}")
        return None

yaml_path = "/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml"
data = diagnose_dataset(yaml_path)

Diagnosing dataset: /content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml
Loaded 67 train images
Directory /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images contains 200 images
Skipped images (200):
  TH_799
  TH_1346
  TH_1466
  TH_1537
  TH_329
  TH_1488
  TH_1299
  TH_681
  TH_1297
  TH_1324
  ... and 190 more


In [ ]:
import os
import glob

# Clear any cache files
cache_patterns = [
    "/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.cache",
    "/content/drive/My Drive/pguard/thermal/thermal_labeled/train.cache",
    "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images.cache"
]
for pattern in cache_patterns:
    for cache_path in glob.glob(pattern):
        if os.path.exists(cache_path):
            os.remove(cache_path)
            print(f"Cleared cache file: {cache_path}")
        else:
            print(f"No cache file found at: {cache_path}")

In [ ]:
from PIL import Image
import os

image_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images"
label_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels"
skipped_samples = ["TH_799", "TH_1346", "TH_1466"]

for base in skipped_samples:
    img_path = os.path.join(image_dir, f"{base}.jpg")  # Adjust extension if needed (.jpeg, .png)
    label_path = os.path.join(label_dir, f"{base}.txt")

    # Check image
    try:
        img = Image.open(img_path)
        img.verify()
        print(f"Image {img_path}: Valid")
        img.close()
    except Exception as e:
        print(f"Image {img_path}: Error - {e}")

    # Check label
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            content = f.read().strip()
            print(f"Label {label_path}: {content if content else 'Empty'}")
    else:
        print(f"Label {label_path}: Not found")

Image /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_799.jpg: Error - [Errno 2] No such file or directory: '/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_799.jpg'
Label /content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels/TH_799.txt: 0 0.482422 0.626736 0.313802 0.086806
Image /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1346.jpg: Error - [Errno 2] No such file or directory: '/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1346.jpg'
Label /content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels/TH_1346.txt: 0 0.787760 0.819444 0.309896 0.114583
Image /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1466.jpg: Error - [Errno 2] No such file or directory: '/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1466.jpg'
Label /content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels/TH_1466.txt: 0 0.684896 0.855035 0.34

In [ ]:
import os

image_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images"
label_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels"

# Count images
images = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Total images: {len(images)}")
print("Sample images:", images[:5])

# Count labels
labels = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
print(f"Total labels: {len(labels)}")
print("Sample labels:", labels[:5])

# Find mismatches
image_bases = {os.path.splitext(f)[0] for f in images}
label_bases = {os.path.splitext(f)[0] for f in labels}
missing_images = label_bases - image_bases
missing_labels = image_bases - label_bases

print(f"Labels without images ({len(missing_images)}):", list(missing_images)[:10])
print(f"Images without labels ({len(missing_labels)}):", list(missing_labels)[:10])

Total images: 200
Sample images: ['TH_1681.png', 'TH_435.png', 'TH_419.png', 'TH_473.png', 'TH_123.png']
Total labels: 200
Sample labels: ['TH_1681.txt', 'TH_435.txt', 'TH_419.txt', 'TH_473.txt', 'TH_123.txt']
Labels without images (0): []
Images without labels (0): []


In [ ]:
from PIL import Image
import os

image_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images"
invalid_images = []
extensions = set()

for img_file in os.listdir(image_dir):
    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
        extensions.add(os.path.splitext(img_file)[1].lower())
        img_path = os.path.join(image_dir, img_file)
        try:
            img = Image.open(img_path)
            img.verify()  # Check for corruption
            img.close()
        except Exception as e:
            invalid_images.append((img_file, str(e)))

print(f"Found image extensions: {extensions}")
if invalid_images:
    print(f"Invalid images ({len(invalid_images)}):")
    for img, error in invalid_images:
        print(f"  {img}: {error}")
else:
    print("All images are readable.")

# Check specific "skipped" images
skipped_samples = ["TH_799", "TH_1346", "TH_1466"]
for base in skipped_samples:
    img_path = os.path.join(image_dir, f"{base}.png")  # Use .png
    try:
        img = Image.open(img_path)
        img.verify()
        print(f"Image {img_path}: Valid")
        img.close()
    except Exception as e:
        print(f"Image {img_path}: Error - {e}")

Found image extensions: {'.png'}
All images are readable.
Image /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_799.png: Valid
Image /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1346.png: Valid
Image /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1466.png: Valid


In [ ]:
from ultralytics.data.utils import check_det_dataset
import os

def diagnose_dataset(yaml_path, verbose=True):
    print(f"Diagnosing dataset: {yaml_path}")
    try:
        data = check_det_dataset(yaml_path)
        train_images = data['train']
        print(f"Loaded {len(train_images)} train images")

        # Check directory contents
        image_dir = os.path.join(os.path.dirname(yaml_path), 'train/images')
        if os.path.isdir(image_dir):
            all_images = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            print(f"Directory {image_dir} contains {len(all_images)} images")

            # Find skipped images
            loaded_bases = {os.path.splitext(os.path.basename(img))[0].lower() for img in train_images}
            all_bases = {os.path.splitext(img)[0].lower() for img in all_images}
            skipped_images = all_bases - loaded_bases
            if skipped_images:
                print(f"Skipped images ({len(skipped_images)}):")
                for base in sorted(list(skipped_images))[:10]:
                    print(f"  {base}")
                    # Check file accessibility
                    img_path = os.path.join(image_dir, next(f for f in all_images if os.path.splitext(f)[0].lower() == base))
                    try:
                        with open(img_path, 'rb') as f:
                            f.read(1)
                        print(f"    {img_path}: File accessible")
                    except Exception as e:
                        print(f"    {img_path}: Error - {e}")
                if len(skipped_images) > 10:
                    print(f"  ... and {len(skipped_images) - 10} more")
            else:
                print("No images skipped.")

        return data
    except Exception as e:
        print(f"Error: {e}")
        return None

yaml_path = "/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml"
data = diagnose_dataset(yaml_path)

Diagnosing dataset: /content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml
Loaded 67 train images
Directory /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images contains 200 images
Skipped images (200):
  th_1003
    /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1003.png: File accessible
  th_1007
    /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1007.png: File accessible
  th_1031
    /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1031.png: File accessible
  th_1044
    /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1044.png: File accessible
  th_1050
    /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1050.png: File accessible
  th_1051
    /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1051.png: File accessible
  th_1057
    /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images/TH_1057.png: 

In [ ]:
import os

label_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels"
skipped_samples = ["TH_1003", "TH_1007", "TH_1031"]

for base in skipped_samples:
    label_path = os.path.join(label_dir, f"{base}.txt")
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            content = f.read().strip()
            print(f"Label {label_path}: {content if content else 'Empty'}")
    else:
        print(f"Label {label_path}: Not found")

Label /content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels/TH_1003.txt: 0 0.456380 0.870660 0.326823 0.105903
Label /content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels/TH_1007.txt: 0 0.632161 0.814236 0.319010 0.100694
Label /content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels/TH_1031.txt: 0 0.177083 0.861111 0.299479 0.083333


In [ ]:
import os
import glob
import shutil

cache_patterns = [
    "/content/drive/My Drive/pguard/thermal/thermal_labeled/*.cache",
    "/content/drive/My Drive/pguard/thermal/thermal_labeled/**/*.cache",
    "/content/*.cache",
    "/root/.cache/ultralytics/*",
    "/tmp/*.cache",
    "/content/drive/My Drive/pguard/thermal/*.cache",
    "*.cache"
]
for pattern in cache_patterns:
    for cache_path in glob.glob(pattern, recursive=True):
        if os.path.exists(cache_path):
            try:
                if os.path.isfile(cache_path):
                    os.remove(cache_path)
                    print(f"Cleared cache file: {cache_path}")
                elif os.path.isdir(cache_path):
                    shutil.rmtree(cache_path)
                    print(f"Cleared cache directory: {cache_path}")
            except Exception as e:
                print(f"Failed to clear {cache_path}: {e}")
        else:
            print(f"No cache file found at: {cache_path}")

In [ ]:
%%writefile train.py

In [ ]:
import os
import yaml

yaml_path = "/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml"
image_dir = "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images"

# Get all image paths
images = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.lower().endswith(('.png',))]
print(f"Found {len(images)} images")

# Update YAML
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = images
temp_yaml = "/content/thermal_labeled_temp.yaml"
with open(temp_yaml, 'w') as f:
    yaml.safe_dump(data, f)
print(f"Created temporary YAML: {temp_yaml}")

# Test with diagnostic
from ultralytics.data.utils import check_det_dataset
try:
    data = check_det_dataset(temp_yaml)
    print(f"Loaded {len(data['train'])} train images")
except Exception as e:
    print(f"Error in diagnostic: {e}")

Found 200 images
Created temporary YAML: /content/thermal_labeled_temp.yaml
Loaded 200 train images


In [ ]:
%%writefile train.py
from ultralytics import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import LOGGER, RANK, TQDM, colorstr, emojis, clean_url
import ultralytics.utils.callbacks.tensorboard as tb_module
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import distributed as dist
import warnings
from torch.nn.utils import spectral_norm
from torchvision.ops import roi_align, box_convert
from functools import partial
import random
import shutil
import time
import pandas as pd
import timeout_decorator
import os
from pathlib import Path
import logging

def seed_everything(seed=2565):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    logging.getLogger('tensorflow').setLevel(logging.ERROR)
    try:
        import tensorflow as tf
        tf.get_logger().setLevel('ERROR')
    except ImportError:
        pass

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class GradientReversalLayer(torch.nn.Module):
    def __init__(self, alpha=1.):
        super(GradientReversalLayer, self).__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

class SpectralLinear(nn.Linear):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)

class SpectralConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        spectral_norm(self, name='weight', n_power_iterations=1, dim=0, eps=1e-12)

class DiscriminatorHead(nn.Module):
    def __init__(self, dim_in, dim_h, dim_o=1):
        super().__init__()
        self.to_flat = nn.Sequential(
            SpectralConv2d(dim_in, dim_h // 2, kernel_size=1),
            nn.Flatten(),
            nn.LazyLinear(dim_h),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.neck = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim_h // 2, dim_h // 2),
                nn.LeakyReLU(0.2, inplace=True),
            ) for _ in range(3)
        ])
        self.head = nn.Sequential(
            SpectralLinear(dim_h // 2 * 4, dim_h // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(dim_h // 2, dim_o, bias=False),
        )

    def forward(self, x):
        x = self.to_flat(x)
        x = x.split(x.shape[1] // 2, dim=1)
        xs = [x[0]]
        for m in self.neck:
            x = m(x[1]) if isinstance(x, tuple) else m(x)
            xs.append(x)
        x = torch.cat(xs, dim=1)
        return self.head(x)

class Discriminator(nn.Module):
    def __init__(self, chs=None, amp=False):
        super().__init__()
        if chs is None:
            chs = [64, 128, 256]
        self.chs = chs
        self.f_len = len(chs)
        self.grl = GradientReversalLayer(alpha=1.0)
        self.amp = amp
        self.p = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(chs[i] if i == 0 else chs[i] * 2, 64, kernel_size=11, stride=2, padding=5, bias=False),
                nn.BatchNorm2d(64),
                nn.SiLU(inplace=True),
                nn.Conv2d(64, 32, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(32),
                nn.SiLU(inplace=True),
                nn.Conv2d(32, chs[i + 1] if i + 1 < len(chs) else chs[i], kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(chs[i + 1] if i + 1 < len(chs) else chs[i]),
                nn.SiLU(inplace=True),
            ) for i in range(len(chs))
        ])
        self.head = DiscriminatorHead(chs[-1], 256)
        self.optim = torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)

    def forward(self, fs: list[torch.tensor]):
        with torch.cuda.amp.autocast(self.amp):
            assert len(fs) == self.f_len, f'Expected {self.f_len} feature maps, got {len(fs)}'
            fs = [self.grl(f) for f in fs]
            x = self.p[0](fs[0])
            for i in range(1, len(fs)):
                x = torch.cat((x, fs[i]), dim=1)
                x = self.p[i](x)
            return self.head(x)

class CustomTrainer(DetectionTrainer):
    def __init__(self, target_domain_data_cfg, labeled_thermal_data=None, *args, **kwargs):
        LOGGER.debug(f"CustomTrainer.__init__ called with kwargs: {kwargs}")
        self.args = kwargs
        self.original_data_cfg = kwargs.get('data', '/content/drive/My Drive/pguard/dataset.yaml')
        LOGGER.info(f"Initializing CustomTrainer with target_domain_data_cfg: {target_domain_data_cfg}, labeled_thermal_data: {labeled_thermal_data}, original_data_cfg: {self.original_data_cfg}")

        # Validate YAML file existence and accessibility
        if not os.path.isfile(self.original_data_cfg):
            raise FileNotFoundError(f"Visible dataset YAML file not found: {self.original_data_cfg}")
        if not os.access(self.original_data_cfg, os.R_OK):
            raise PermissionError(f"Cannot read visible dataset YAML file: {self.original_data_cfg}")

        # Load visible dataset (source domain)
        try:
            self.data = check_det_dataset(self.original_data_cfg)
            base_path = Path(self.data['path'])
            expected_train_path = base_path / 'train' / 'images'
            self.data['train'] = str(expected_train_path)
            LOGGER.info(f"Visible data train path set to: {self.data['train']}")
            if not os.path.exists(self.data['train']):
                raise FileNotFoundError(f"Visible dataset train directory does not exist: {self.data['train']}")
            valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')
            train_images = [str(f) for f in Path(self.data['train']).glob('*') if f.suffix.lower() in valid_extensions]
            LOGGER.info(f"Visible data validated with {len(train_images)} train images (scanned)")
            if len(train_images) < 100:
                LOGGER.warning(f"Low train image count ({len(train_images)}) for visible dataset. Expected >1000.")
            label_dir = Path(self.data['train']).parent / 'labels'
            # Remove cache to force re-scan
            cache_path = str(label_dir / 'labels.cache')
            if os.path.exists(cache_path):
                LOGGER.info(f"Removing cache file for visible dataset: {cache_path}")
                os.remove(cache_path)
            self._validate_labels(self.data['train'], label_dir, "visible", require_labels=True)
        except Exception as e:
            raise RuntimeError(f"Failed to validate visible dataset '{clean_url(self.original_data_cfg)}': {e}") from e

        # Load target domain (thermal) dataset
        if not os.path.isfile(target_domain_data_cfg):
            raise FileNotFoundError(f"Thermal dataset YAML file not found: {target_domain_data_cfg}")
        if not os.access(target_domain_data_cfg, os.R_OK):
            raise PermissionError(f"Cannot read thermal dataset YAML file: {target_domain_data_cfg}")
        try:
            self.t_data = check_det_dataset(target_domain_data_cfg)
            base_path = Path(self.t_data['path'])
            expected_train_path = base_path / 'train' / 'images'
            self.t_data['train'] = str(expected_train_path)
            LOGGER.info(f"Thermal data train path set to: {self.t_data['train']}")
            if not os.path.exists(self.t_data['train']):
                raise FileNotFoundError(f"Thermal dataset train directory does not exist: {self.t_data['train']}")
            train_images = [str(f) for f in Path(self.t_data['train']).glob('*') if f.suffix.lower() in valid_extensions]
            LOGGER.info(f"Target domain data validated with {len(train_images)} train images (scanned)")
            if len(train_images) < 100:
                LOGGER.warning(f"Low train image count ({len(train_images)}) for thermal dataset. Expected >1000.")
            label_dir = Path(self.t_data['train']).parent / 'labels'
            # Remove cache to force re-scan
            cache_path = str(label_dir / 'labels.cache')
            if os.path.exists(cache_path):
                LOGGER.info(f"Removing cache file for thermal dataset: {cache_path}")
                os.remove(cache_path)
            self._validate_labels(self.t_data['train'], label_dir, "thermal", require_labels=False)

            self.t_trainset, self.t_testset = self.get_dataset(dataset_type='thermal', check_labels=False)
            LOGGER.info(f"Thermal dataset initialized with {len(self.t_trainset.img_files)} samples")

            self._include_all_thermal_images(self.t_data['train'], label_dir)
            LOGGER.info(f"Thermal dataset updated with {len(self.t_trainset.img_files)} images")
        except Exception as e:
            raise RuntimeError(emojis(f"Dataset '{clean_url(target_domain_data_cfg)}' error ❌ {e}")) from e

        # Load labeled thermal dataset
        self.labeled_thermal_data = None
        if labeled_thermal_data:
            if not os.path.isfile(labeled_thermal_data):
                LOGGER.warning(f"Labeled thermal dataset YAML file not found: {labeled_thermal_data}")
            elif not os.access(labeled_thermal_data, os.R_OK):
                LOGGER.warning(f"Cannot read labeled thermal dataset YAML file: {labeled_thermal_data}")
            else:
                try:
                    self.labeled_thermal_data = check_det_dataset(labeled_thermal_data)
                    base_path = Path(self.labeled_thermal_data['path'])
                    expected_train_path = base_path / 'train' / 'images'
                    self.labeled_thermal_data['train'] = str(expected_train_path)
                    LOGGER.info(f"Labeled thermal data train path set to: {self.labeled_thermal_data['train']}")
                    train_images = [str(f) for f in Path(self.labeled_thermal_data['train']).glob('*') if f.suffix.lower() in valid_extensions]
                    LOGGER.info(f"Labeled thermal data validated: {len(train_images)} train images (scanned)")
                    label_dir = Path(self.labeled_thermal_data['train']).parent / 'labels'
                    # Remove cache to force re-scan
                    cache_path = str(label_dir / 'labels.cache')
                    if os.path.exists(cache_path):
                        LOGGER.info(f"Removing cache file for labeled thermal dataset: {cache_path}")
                        os.remove(cache_path)
                    self._validate_labels(self.labeled_thermal_data['train'], label_dir, "labeled thermal")
                except Exception as e:
                    LOGGER.error(f"Failed to validate labeled thermal dataset: {e}")
                    self.labeled_thermal_data = None

        super(CustomTrainer, self).__init__(*args, **kwargs)

        self.trainset, self.testset = self.get_dataset(dataset_type='visible')
        LOGGER.info(f"Visible dataset loaded with {len(self.trainset.img_files)} samples")

        self.labeled_thermal_trainset = None
        if self.labeled_thermal_data:
            try:
                self.labeled_thermal_trainset = self.get_dataset(dataset_type='labeled_thermal')[0]
                LOGGER.info(f"Labeled thermal dataset loaded with {len(self.labeled_thermal_trainset.img_files)} samples")
            except Exception as e:
                LOGGER.error(f"Failed to initialize labeled_thermal_trainset: {e}")
                self.labeled_thermal_trainset = None

        self.t_iter = None
        self.t_train_loader = None
        self.labeled_thermal_iter = None
        self.labeled_thermal_loader = None

        self.visible_augs = {
            'hsv_h': 0.015,
            'hsv_s': 0.9,
            'hsv_v': 0.6,
            'degrees': 10.0,
            'translate': 0.1,
            'scale': 0.5,
            'mosaic': True
        }
        self.thermal_augs = {
            'hsv_h': 0.0,
            'hsv_s': 0.0,
            'hsv_v': 0.2,
            'degrees': 5.0,
            'translate': 0.05,
            'scale': 0.2,
            'mosaic': False
        }

        self.model_hook_handler = []
        self.model_hook_layer_idx: list[int] = [2, 4, 6]
        self.roi_size = list(reversed([20 * 2 ** i for i in range(len(self.model_hook_layer_idx))]))
        self.model_hooked_features: None | list[torch.tensor] = None
        self.discriminator_model = None
        self.additional_models = []
        self.tensorboard_writer = None
        self.thermal_metrics = []
        self.add_callback('on_train_start', self.init_helper_model)
        self.add_callback('on_train_start', self.init_tensorboard_writer)
        self.add_callback('teardown', self.close_tensorboard_writer)

    def _validate_labels(self, train_images, label_dir, dataset_name, require_labels=True):
        missing_labels = []
        empty_labels = []
        valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')
        if isinstance(train_images, str):
            image_paths = [os.path.join(train_images, f) for f in os.listdir(train_images) if f.lower().endswith(valid_extensions)]
        else:
            image_paths = train_images
        for img_path in image_paths:
            label_path = label_dir / (Path(img_path).stem + '.txt')
            if not label_path.exists():
                missing_labels.append(str(label_path))
            elif label_path.stat().st_size == 0:
                empty_labels.append(str(label_path))
        if missing_labels and require_labels:
            LOGGER.warning(f"Missing {len(missing_labels)} label files for {dataset_name} dataset: {missing_labels[:5]}...")
        elif missing_labels:
            LOGGER.info(f"Found {len(missing_labels)} missing label files for {dataset_name} dataset, proceeding without labels: {missing_labels[:5]}...")
        if empty_labels and require_labels:
            LOGGER.warning(f"Empty {len(empty_labels)} label files for {dataset_name} dataset: {empty_labels[:5]}...")
        elif empty_labels:
            LOGGER.info(f"Found {len(empty_labels)} empty label files for {dataset_name} dataset, treating as no labels: {empty_labels[:5]}...")

    def _include_all_thermal_images(self, train_images, label_dir):
        if isinstance(train_images, str) and os.path.isdir(train_images):
            image_dir = Path(train_images)
        else:
            image_dir = Path(self.t_data['path']) / 'train' / 'images'
            LOGGER.warning(f"Invalid train_images: {train_images}. Falling back to {image_dir}")

        if not image_dir.exists():
            LOGGER.warning(f"Thermal image directory does not exist: {image_dir}. Proceeding with empty thermal dataset.")
            self.t_trainset.img_files = []
            self.t_trainset.labels = []
            return
        if not os.access(image_dir, os.R_OK):
            raise PermissionError(f"Cannot read thermal image directory: {image_dir}")

        valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')
        all_images = sorted([str(f) for f in image_dir.glob('*') if f.suffix.lower() in valid_extensions])
        LOGGER.info(f"Scanned {len(all_images)} thermal images in {image_dir}")
        LOGGER.debug(f"First few images: {all_images[:5]}")

        all_files = sorted(os.listdir(image_dir))
        LOGGER.debug(f"All files in {image_dir}: {all_files[:10]}")

        if not all_images:
            LOGGER.warning(f"No images found in thermal directory: {image_dir}. Proceeding with empty thermal dataset.")
            self.t_trainset.img_files = []
            self.t_trainset.labels = []
            return

        self.t_trainset.img_files = all_images
        self.t_trainset.labels = []
        for img in all_images:
            label_path = label_dir / (Path(img).stem + '.txt')
            if label_path.exists() and label_path.stat().st_size > 0:
                with open(label_path, 'r') as f:
                    labels = [line.strip().split() for line in f.readlines()]
                    labels = [[float(x) for x in label] for label in labels]
            else:
                labels = []
            self.t_trainset.labels.append(labels)
        LOGGER.info(f"Updated t_trainset to include {len(all_images)} thermal images, with {len([l for l in self.t_trainset.labels if not l])} images having no labels.")

    def init_helper_model(self, *args, **kwargs):
        LOGGER.info("Initializing helper model")
        self.discriminator_model = Discriminator(amp=self.amp).to(self.device)
        self.additional_models.append(self.discriminator_model)

    def init_tensorboard_writer(self, *args, **kwargs):
        if RANK in (-1, 0):
            self.tensorboard_writer = tb_module.SummaryWriter(log_dir=self.save_dir)
            LOGGER.info(f"TensorBoard writer initialized at {self.save_dir}")

    def close_tensorboard_writer(self, *args, **kwargs):
        if self.tensorboard_writer is not None:
            self.tensorboard_writer.close()
            LOGGER.info("TensorBoard writer closed")

    def save_thermal_metrics_to_csv(self):
        if self.thermal_metrics:
            df = pd.DataFrame(self.thermal_metrics)
            csv_path = os.path.join(self.save_dir, 'thermal_results.csv')
            df.to_csv(csv_path, index=False)
            LOGGER.info(f"Thermal metrics saved to {csv_path}")

    def get_dataset(self, dataset_type='visible', check_labels=True):
        from ultralytics.data.dataset import YOLODataset
        if dataset_type == 'visible':
            data = self.data
            require_labels = True
        elif dataset_type == 'thermal':
            data = self.t_data
            require_labels = False
        elif dataset_type == 'labeled_thermal':
            data = self.labeled_thermal_data
            require_labels = True
        else:
            raise ValueError(f"Invalid dataset_type: {dataset_type}")

        base_path = Path(data['path'])
        train_key = data.get('train', '')

        if not train_key:
            image_dir = base_path / 'train' / 'images'
            LOGGER.warning(f"train_key is empty for {dataset_type} dataset. Falling back to {image_dir}")
        elif isinstance(train_key, str):
            train_path = Path(train_key)
            if train_path.is_absolute():
                image_dir = train_path
            else:
                image_dir = base_path / train_path
            if image_dir.name != 'images':
                image_dir = image_dir / 'images'
                LOGGER.debug(f"Adjusted image_dir to include 'images': {image_dir}")
        else:
            image_dir = base_path / 'train' / 'images'
            LOGGER.warning(f"Invalid train_key for {dataset_type} dataset: {train_key}. Falling back to {image_dir}")

        if not image_dir.exists():
            raise FileNotFoundError(f"Image directory for {dataset_type} dataset does not exist: {image_dir}")
        if not os.access(image_dir, os.R_OK):
            raise PermissionError(f"Cannot read image directory for {dataset_type} dataset: {image_dir}")

        valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')
        all_images = sorted([str(f) for f in image_dir.glob('*') if f.suffix.lower() in valid_extensions])
        LOGGER.info(f"Scanned {len(all_images)} images for {dataset_type} dataset in {image_dir}")
        LOGGER.debug(f"First few images for {dataset_type}: {all_images[:5]}")

        all_files = sorted(os.listdir(image_dir))
        LOGGER.debug(f"All files in {image_dir}: {all_files[:10]}")

        if not all_images:
            if dataset_type == 'visible':
                raise FileNotFoundError(f"No images found in {dataset_type} directory: {image_dir}")
            else:
                LOGGER.warning(f"No images found in {dataset_type} directory: {image_dir}. Proceeding with empty dataset.")
                return YOLODataset(
                    img_path=str(image_dir),
                    imgsz=self.args.get('imgsz', 640),
                    data=data,
                    augment=True,
                    cache=self.args.get('cache', False),
                    prefix=dataset_type,
                    rect=False
                ), None

        # Create dataset with minimal validation for thermal dataset
        dataset = YOLODataset(
            img_path=str(image_dir),
            imgsz=self.args.get('imgsz', 640),
            data=data,
            augment=True,
            cache=self.args.get('cache', False),
            prefix=dataset_type,
            rect=False,
            single_cls=self.args.get('single_cls', False),
        )

        # Override img_files and labels to include all images
        dataset.img_files = all_images
        dataset.labels = []
        label_dir = image_dir.parent / 'labels'
        for img in all_images:
            label_path = label_dir / (Path(img).stem + '.txt')
            if label_path.exists() and label_path.stat().st_size > 0:
                with open(label_path, 'r') as f:
                    labels = [line.strip().split() for line in f.readlines()]
                    labels = [[float(x) for x in label] for label in labels]
            else:
                labels = []
            dataset.labels.append(labels)

        LOGGER.info(f"Loaded {len(dataset.img_files)} images for {dataset_type} dataset, with {len([l for l in dataset.labels if not l])} images having no labels.")
        if len(dataset.img_files) != len(all_images):
            LOGGER.warning(f"Mismatch in image count after override for {dataset_type} dataset: expected {len(all_images)}, got {len(dataset.img_files)}")

        # If labels are required, filter out images without labels
        if require_labels:
            filtered_img_files = []
            filtered_labels = []
            for img, lbl in zip(dataset.img_files, dataset.labels):
                if lbl:  # Only include images with non-empty labels
                    filtered_img_files.append(img)
                    filtered_labels.append(lbl)
            dataset.img_files = filtered_img_files
            dataset.labels = filtered_labels
            LOGGER.info(f"After filtering for required labels, {dataset_type} dataset has {len(dataset.img_files)} images.")

        return dataset, None

    def get_dataloader(self, dataset, batch_size, rank=-1, mode='train'):
        if mode == 'train':
            if dataset == self.trainset:
                augs = self.visible_augs
            elif dataset == self.t_trainset:
                augs = self.thermal_augs
            elif dataset == self.labeled_thermal_trainset:
                augs = self.thermal_augs
            else:
                augs = {}
            if hasattr(dataset, 'hsv_h'):
                dataset.hsv_h = augs.get('hsv_h', 0.0)
                dataset.hsv_s = augs.get('hsv_s', 0.0)
                dataset.hsv_v = augs.get('hsv_v', 0.0)
                dataset.degrees = augs.get('degrees', 0.0)
                dataset.translate = augs.get('translate', 0.0)
                dataset.scale = augs.get('scale', 0.0)
                dataset.mosaic = augs.get('mosaic', True)
        return super().get_dataloader(dataset, batch_size, rank, mode)

    def get_t_batch(self):
        if self.t_trainset is None or not self.t_trainset.img_files:
            LOGGER.warning("No thermal dataset available. Skipping thermal batch.")
            return None
        if self.t_iter is None:
            LOGGER.info("Initializing target domain data iterator")
            self.t_train_loader = self.get_dataloader(self.t_trainset, batch_size=self.batch_size, rank=RANK, mode='train')
            self.t_iter = iter(self.t_train_loader)
        try:
            batch = next(self.t_iter)
        except StopIteration:
            LOGGER.info("Resetting target domain data iterator")
            self.t_iter = iter(self.t_train_loader)
            batch = next(self.t_iter)
        return batch

    def get_labeled_thermal_batch(self):
        if self.labeled_thermal_data is None or self.labeled_thermal_trainset is None:
            return None
        if self.labeled_thermal_iter is None:
            LOGGER.info("Initializing labeled thermal data iterator")
            self.labeled_thermal_loader = self.get_dataloader(self.labeled_thermal_trainset, batch_size=self.batch_size, rank=RANK, mode='train')
            self.labeled_thermal_iter = iter(self.labeled_thermal_loader)
        try:
            batch = next(self.labeled_thermal_iter)
        except StopIteration:
            LOGGER.info("Resetting labeled thermal data iterator")
            self.labeled_thermal_iter = iter(self.labeled_thermal_loader)
            batch = next(self.labeled_thermal_iter)
        return batch

    def activate_hook(self, layer_indices: list[int] = None):
        if layer_indices is not None:
            self.model_hook_layer_idx = layer_indices
        self.model_hooked_features = [None for _ in self.model_hook_layer_idx]
        self.model_hook_handler = \
            [self.model.model[l].register_forward_hook(self.hook_fn(i)) for i, l in enumerate(self.model_hook_layer_idx)]

    def deactivate_hook(self):
        if self.model_hook_handler is not None:
            for hook in self.model_hook_handler:
                hook.remove()
            self.model_hooked_features = None
            self.model_hook_handler = []

    def hook_fn(self, hook_idx):
        def hook(m, i, o):
            self.model_hooked_features[hook_idx] = o
        return hook

    def get_dis_output_from_hooked_features(self, batch):
        if self.model_hooked_features is not None:
            bbox_batch_idx = batch['batch_idx'].unsqueeze(-1)
            bbox = batch['bboxes']
            if bbox.numel() == 0:
                dummy_bbox = torch.tensor([[0.5, 0.5, 0.5, 0.5]], device=bbox.device).expand(batch['batch_idx'].size(0), -1)
                bbox = dummy_bbox
            bbox = box_convert(bbox, 'cxcywh', 'xyxy')
            rois = []
            for fidx, f in enumerate(self.model_hooked_features):
                f_bbox = bbox * f.shape[-1]
                f_bbox = torch.cat((bbox_batch_idx, f_bbox), dim=-1)
                f_roi = roi_align(f, f_bbox.to(f.device), output_size=self.roi_size[fidx], aligned=True)
                rois.append(f_roi)
            dis_output = self.discriminator_model(rois)
            return dis_output
        else:
            return None

    def optimizer_step(self, optims: None | list[torch.optim.Optimizer] = None):
        self.scaler.unscale_(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.unscale_(o)
        max_norm = 10.0
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=max_norm)
        if len(self.additional_models) > 0:
            for m in self.additional_models:
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=max_norm * 2)
        self.scaler.step(self.optimizer)
        if optims is not None:
            for o in optims:
                if o.param_groups[0]['params'][0].grad is not None:
                    self.scaler.step(o)
        self.scaler.update()
        self.optimizer.zero_grad()
        if optims is not None:
            for o in optims:
                o.zero_grad()
        if self.ema:
            self.ema.update(self.model)

    def _do_train(self, world_size=1):
        if world_size > 1:
            self._setup_ddp(world_size)
        self._setup_train(world_size)

        self.epoch_time = None
        self.epoch_time_start = time.time()
        self.train_time_start = time.time()
        nb = len(self.train_loader)
        nw = max(round(self.args.get('warmup_epochs', 3.0) * nb), 100) if self.args.get('warmup_epochs', 3.0) > 0 else -1
        last_opt_step = -1
        self.run_callbacks('on_train_start')
        LOGGER.info(f'Image sizes {self.args.get("imgsz", 640)} train, {self.args.get("imgsz", 640)} val\n'
                    f'Using {self.train_loader.num_workers * (world_size or 1)} dataloader workers\n'
                    f"Logging results to {colorstr('bold', self.save_dir)}\n"
                    f'Starting training for {self.args.get("epochs", 40)} epochs...')
        if self.args.get('close_mosaic', 10):
            base_idx = (self.args.get('epochs', 40) - self.args.get('close_mosaic', 10)) * nb
            self.plot_idx.extend([base_idx, base_idx + 1, base_idx + 2])
        self.activate_hook()

        save_dir_drive = "/content/drive/My Drive/pguard/checkpoints/visible_to_thermal_semisupervised"
        os.makedirs(save_dir_drive, exist_ok=True)

        for epoch in range(self.start_epoch, self.args.get('epochs', 40)):
            LOGGER.info(f"Starting epoch {epoch + 1}")
            self.epoch = epoch
            self.run_callbacks('on_train_epoch_start')
            self.model.train()
            if RANK != -1:
                self.train_loader.sampler.set_epoch(epoch)
            pbar = enumerate(self.train_loader)
            if epoch == (self.args.get('epochs', 40) - self.args.get('close_mosaic', 10)):
                LOGGER.info('Closing dataloader mosaic')
                if hasattr(self.train_loader.dataset, 'mosaic'):
                    self.train_loader.dataset.mosaic = False
                if hasattr(self.train_loader.dataset, 'close_mosaic'):
                    self.train_loader.dataset.close_mosaic(hyp=self.args)
                self.train_loader.reset()
            if RANK in (-1, 0):
                LOGGER.info(self.progress_string())
                pbar = TQDM(pbar, total=nb)
            self.tloss = None
            self.optimizer.zero_grad()
            for i, batch in pbar:
                self.run_callbacks('on_train_batch_start')
                ni = i + nb * epoch
                if ni <= nw:
                    xi = [0, nw]
                    self.accumulate = max(1, np.interp(ni, xi, [1, self.args.get('nbs', 64) / self.batch_size]).round())
                    for j, x in enumerate(self.optimizer.param_groups):
                        x['lr'] = np.interp(
                            ni, xi, [self.args.get('warmup_bias_lr', 0.1) if j == 0 else 0.0, x['initial_lr'] * self.lf(epoch)])
                        if 'momentum' in x:
                            x['momentum'] = np.interp(ni, xi, [self.args.get('warmup_momentum', 0.8), self.args.get('momentum', 0.937)])
                try:
                    with torch.cuda.amp.autocast(self.amp):
                        batch = self.preprocess_batch(batch)
                        self.loss, self.loss_items = self.model(batch)
                        if RANK != -1:
                            self.loss *= world_size
                        self.tloss = (self.tloss * i + self.loss_items) / (i + 1) if self.tloss is not None \
                            else self.loss_items
                        source_critics = self.get_dis_output_from_hooked_features(batch)

                        t_batch = self.get_t_batch()
                        if t_batch:
                            t_batch = self.preprocess_batch(t_batch)
                            target_critics = self.get_dis_output_from_hooked_features(t_batch)
                        else:
                            target_critics = torch.tensor(0.0, device=self.device)

                        self_loss_scalar = self.loss if isinstance(self.loss, torch.Tensor) and self.loss.numel() == 1 else self.loss.mean() if isinstance(self.loss, torch.Tensor) else self.loss

                        if 6 < epoch < self.args.get('epochs', 40) - 10:
                            threshold = 10
                            loss_d_source = F.relu(torch.ones_like(source_critics) * threshold + source_critics).mean()
                            loss_d_target = F.relu(torch.ones_like(target_critics) * threshold - target_critics).mean()
                            loss_d = (loss_d_source + loss_d_target) * 0.5
                            alpha_d = np.interp(epoch, [6, 10, self.args.get('epochs', 40) - 10, self.args.get('epochs', 40)], [0.0, 0.5, 0.5, 0.0])
                        else:
                            loss_d = torch.tensor(0.0, device=self.device)
                            alpha_d = 0.0

                        labeled_loss = 0.0
                        if self.labeled_thermal_data and self.labeled_thermal_trainset is not None:
                            labeled_batch = self.get_labeled_thermal_batch()
                            if labeled_batch:
                                labeled_batch = self.preprocess_batch(labeled_batch)
                                labeled_loss, _ = self.model(labeled_batch)
                                labeled_loss = labeled_loss.mean()
                            alpha_l = min(0.05, (epoch + 1) / 30.0)
                        else:
                            alpha_l = 0.0

                        self.loss = self_loss_scalar + alpha_d * loss_d + alpha_l * labeled_loss

                        if RANK in (-1, 0) and self.tensorboard_writer is not None:
                            self.tensorboard_writer.add_scalar('train/self_loss', self_loss_scalar.item(), ni)
                            self.tensorboard_writer.add_scalar('train/loss_d', loss_d.item(), ni)
                            self.tensorboard_writer.add_scalar('train/weighted_loss_d', (alpha_d * loss_d).item(), ni)
                            self.tensorboard_writer.add_scalar('train/labeled_loss', labeled_loss.item(), ni)
                            self.tensorboard_writer.add_scalar('train/weighted_labeled_loss', (alpha_l * labeled_loss).item(), ni)
                            self.tensorboard_writer.add_scalar('train/alpha_d', alpha_d, ni)
                            self.tensorboard_writer.add_scalar('train/alpha_l', alpha_l, ni)
                            self.tensorboard_writer.add_scalar('train/critic-source', source_critics.mean().item(), ni)
                            self.tensorboard_writer.add_scalar('train/critic-target', target_critics.mean().item(), ni)
                            mem_allocated = torch.cuda.memory_allocated() / 1E9 if torch.cuda.is_available() else 0
                            mem_reserved = torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0
                            self.tensorboard_writer.add_scalar('train/mem_allocated', mem_allocated, ni)
                            self.tensorboard_writer.add_scalar('train/mem_reserved', mem_reserved, ni)

                    self.scaler.scale(self.loss).backward()
                    if ni - last_opt_step >= self.accumulate:
                        self.optimizer_step(optims=[self.discriminator_model.optim])
                        last_opt_step = ni
                except Exception as e:
                    LOGGER.error(f"Error during training batch {i}: {e}")
                    continue

                mem = f'{torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0:.3g}G'
                loss_len = self.tloss.shape[0] if len(self.tloss.size()) else 1
                losses = self.tloss if loss_len > 1 else torch.unsqueeze(self.tloss, 0)
                if RANK in (-1, 0):
                    mem_allocated = torch.cuda.memory_allocated() / 1E9 if torch.cuda.is_available() else 0
                    mem_reserved = torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0
                    pbar.set_description(
                        ('%11s' * 2 + '%11.4g' * (2 + loss_len) + ' Mem: %.3gG/%.3gG') %
                        (f'{epoch + 1}/{self.args.get("epochs", 40)}', mem, *losses, batch['cls'].shape[0], batch['img'].shape[-1],
                         mem_allocated, mem_reserved))
                    self.run_callbacks('on_batch_end')
                    if self.args.get('plots', True) and ni in self.plot_idx:
                        self.plot_training_samples(batch, ni)
                self.run_callbacks('on_train_batch_end')

            self.lr = {f'lr/pg{ir}': x['lr'] for ir, x in enumerate(self.optimizer.param_groups)}
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                self.scheduler.step()
            self.run_callbacks('on_train_epoch_end')
            if RANK in (-1, 0):
                self.ema.update_attr(self.model, include=['yaml', 'nc', 'args', 'names', 'stride', 'class_weights'])
                final_epoch = (epoch + 1 == self.args.get('epochs', 40)) or self.stopper.possible_stop
                if self.args.get('val', True) or final_epoch:
                    self.metrics, self.fitness = self.validate()
                self.save_metrics(metrics={**self.label_loss_items(self.tloss), **self.metrics, **self.lr})
                self.stop = self.stopper(epoch + 1, self.fitness)
                if self.args.get('save', True) or (epoch + 1 == self.args.get('epochs', 40)):
                    self.deactivate_hook()
                    self.save_model()
                    source_dir = os.path.join(str(self.save_dir), "weights")
                    for checkpoint in ["last.pt", "best.pt"]:
                        source_path = os.path.join(source_dir, checkpoint)
                        dest_path = os.path.join(save_dir_drive, f"{checkpoint.replace('.pt', '')}_epoch_{epoch + 1}.pt")
                        if os.path.exists(source_path):
                            shutil.copyfile(source_path, dest_path)
                            LOGGER.info(f"Saved {checkpoint} for epoch {epoch + 1} to {dest_path}")
                    self.run_callbacks('on_model_save')
                    self.activate_hook()

                LOGGER.info(f"\n=== Thermal Dataset Evaluation After Epoch {epoch + 1} ===")
                LOGGER.info(f"Evaluating on thermal dataset: {self.t_data['yaml_file']}")
                thermal_eval_model = YOLO(os.path.join(str(self.save_dir), "weights", "last.pt"))
                thermal_metrics = {'epoch': epoch + 1, 'precision': 0.0, 'recall': 0.0, 'map50': 0.0, 'map50_95': 0.0}
                try:
                    thermal_results = thermal_eval_model.val(
                        data=self.t_data['yaml_file'],
                        split="val",
                        augment=False,
                        verbose=True
                    )
                    if thermal_results and len(thermal_results.box.p) > 0:
                        thermal_metrics = {
                            'epoch': epoch + 1,
                            'precision': float(thermal_results.box.p[0]),
                            'recall': float(thermal_results.box.r[0]),
                            'map50': float(thermal_results.box.map50),
                            'map50_95': float(thermal_results.box.map)
                        }
                        self.thermal_metrics.append(thermal_metrics)
                        if self.tensorboard_writer:
                            self.tensorboard_writer.add_scalar('thermal/precision', thermal_metrics['precision'], epoch)
                            self.tensorboard_writer.add_scalar('thermal/recall', thermal_metrics['recall'], epoch)
                            self.tensorboard_writer.add_scalar('thermal/map50', thermal_metrics['map50'], epoch)
                            self.tensorboard_writer.add_scalar('thermal/map50_95', thermal_metrics['map50_95'], epoch)
                    else:
                        LOGGER.warning("No valid metrics computed for thermal dataset. Check dataset labels in val split.")
                except Exception as e:
                    LOGGER.warning(f"Failed to evaluate on thermal dataset: {e}")
                LOGGER.info(f"\nThermal Evaluation Results After Epoch {epoch + 1}:")
                LOGGER.info(f"Precision: {thermal_metrics['precision']:.3f}")
                LOGGER.info(f"Recall: {thermal_metrics['recall']:.3f}")
                LOGGER.info(f"mAP@50: {thermal_metrics['map50']:.3f}")
                LOGGER.info(f"mAP@50:95: {thermal_metrics['map50_95']:.3f}")
                LOGGER.info(f"=== Thermal Evaluation Complete for Epoch {epoch + 1} ===\n")

            tnow = time.time()
            self.epoch_time = tnow - self.epoch_time_start
            self.epoch_time_start = tnow
            self.run_callbacks('on_fit_epoch_end')
            torch.cuda.empty_cache()
            if RANK != -1:
                broadcast_list = [self.stop if RANK == 0 else None]
                dist.broadcast_object_list(broadcast_list, 0)
                if RANK != -1:
                    self.stop = broadcast_list[0]
            if self.stop:
                break

        if RANK in (-1, 0):
            LOGGER.info(f'\n{epoch - self.start_epoch + 1} epochs completed in '
                        f'{(time.time() - self.train_time_start) / 3600:.3f} hours.')
            self.final_eval()
            if self.args.get('plots', True):
                self.plot_metrics()
            if self.thermal_metrics:
                LOGGER.info("\n=== Average Thermal Dataset Evaluation Metrics Across All Epochs ===")
                avg_precision = np.mean([m['precision'] for m in self.thermal_metrics])
                avg_recall = np.mean([m['recall'] for m in self.thermal_metrics])
                avg_map50 = np.mean([m['map50'] for m in self.thermal_metrics])
                avg_map50_95 = np.mean([m['map50_95'] for m in self.thermal_metrics])
                LOGGER.info(f"Average Precision: {avg_precision:.3f}")
                LOGGER.info(f"Average Recall: {avg_recall:.3f}")
                LOGGER.info(f"Average mAP@50: {avg_map50:.3f}")
                LOGGER.info(f"Average mAP@50:95: {avg_map50_95:.3f}")
                LOGGER.info("=== Average Thermal Evaluation Complete ===\n")
            self.save_thermal_metrics_to_csv()
            self.run_callbacks('on_train_end')
        self.deactivate_hook()
        torch.cuda.empty_cache()
        self.run_callbacks('teardown')

@timeout_decorator.timeout(300)
def validate_dataset(yaml_path):
    try:
        if not os.path.isfile(yaml_path):
            raise FileNotFoundError(f"YAML file not found: {yaml_path}")
        if not os.access(yaml_path, os.R_OK):
            raise PermissionError(f"Cannot read YAML file: {yaml_path}")
        data_info = check_det_dataset(yaml_path)
        base_path = Path(data_info['path'])
        expected_train_path = base_path / 'train' / 'images'
        data_info['train'] = str(expected_train_path)
        if not os.path.exists(data_info['train']):
            raise FileNotFoundError(f"Train directory does not exist: {data_info['train']}")
        valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')
        train_images = [str(f) for f in Path(data_info['train']).glob('*') if f.suffix.lower() in valid_extensions]
        val_images = data_info.get('val', '')
        if val_images:
            val_images = str(Path(val_images) if Path(val_images).is_absolute() else base_path / val_images)
            val_images = [str(f) for f in Path(val_images).glob('*') if f.suffix.lower() in valid_extensions]
        LOGGER.info(f"Dataset {yaml_path}: train images={len(train_images)} (scanned), val images={len(val_images)} (scanned)")
        if len(train_images) < 100 and 'labeled' not in yaml_path:
            LOGGER.warning(f"Low train image count ({len(train_images)}) for {yaml_path}. Expected >1000. Verify directory contents.")
        image_dir = data_info['train']
        label_dir = str(Path(image_dir).parent / 'labels')
        missing_labels = []
        empty_labels = []
        for img in train_images:
            label_path = os.path.join(label_dir, Path(img).stem + '.txt')
            if not os.path.exists(label_path):
                missing_labels.append(label_path)
            elif os.path.getsize(label_path) == 0:
                empty_labels.append(label_path)
        if missing_labels and 'thermal.yaml' not in yaml_path:
            LOGGER.warning(f"Missing {len(missing_labels)} label files for {yaml_path}: {missing_labels[:5]}...")
        elif missing_labels:
            LOGGER.info(f"Found {len(missing_labels)} missing label files for {yaml_path}, proceeding without labels for domain adaptation: {missing_labels[:5]}...")
        if empty_labels and 'thermal.yaml' not in yaml_path:
            LOGGER.warning(f"Empty {len(empty_labels)} label files for {yaml_path}: {empty_labels[:5]}...")
        elif empty_labels:
            LOGGER.info(f"Found {len(empty_labels)} empty label files for {yaml_path}, treating as no labels: {empty_labels[:5]}...")
        cache_path = str(Path(image_dir).parent / 'labels.cache')
        if os.path.exists(cache_path):
            LOGGER.info(f"Cache file found: {cache_path}, removing to force re-scan...")
            os.remove(cache_path)
        return data_info
    except Exception as e:
        LOGGER.error(f"Dataset validation failed for {yaml_path}: {e}")
        raise

def main():
    seed = 2565
    kwargs = {
        'model': 'yolov8s.yaml',
        'data': '/content/drive/My Drive/pguard/dataset.yaml',
        'epochs': 40,
        'patience': 10,
        'batch': 16,
        'imgsz': 640,
        'save': True,
        'save_period': -1,
        'cache': False,
        'device': None,
        'workers': 8,
        'project': None,
        'name': 'visible_to_thermal_semisupervised_adversarial',
        'exist_ok': False,
        'pretrained': '/content/best.pt',
        'optimizer': 'auto',
        'verbose': True,
        'seed': seed,
        'deterministic': True,
        'single_cls': False,
        'rect': False,
        'cos_lr': True,
        'close_mosaic': 10,
        'resume': False,
        'amp': True,
        'fraction': 1.0,
        'profile': False,
        'freeze': None,
        'multi_scale': False,
        'overlap_mask': True,
        'mask_ratio': 4,
        'dropout': 0.0,
        'val': True,
        'split': 'val',
        'save_json': False,
        'conf': None,
        'iou': 0.7,
        'max_det': 300,
        'half': False,
        'dnn': False,
        'plots': True,
        'source': None,
        'vid_stride': 1,
        'stream_buffer': False,
        'visualize': False,
        'augment': False,
        'agnostic_nms': False,
        'classes': None,
        'retina_masks': False,
        'embed': None,
        'show': False,
        'save_frames': False,
        'save_txt': False,
        'save_conf': False,
        'save_crop': False,
        'show_labels': True,
        'show_conf': True,
        'show_boxes': True,
        'line_width': None,
        'format': 'torchscript',
        'keras': False,
        'optimize': False,
        'int8': False,
        'dynamic': False,
        'simplify': True,
        'opset': None,
        'workspace': None,
        'nms': False,
        'lr0': 0.01,
        'lrf': 0.01,
        'momentum': 0.937,
        'weight_decay': 0.0005,
        'warmup_epochs': 3.0,
        'warmup_momentum': 0.8,
        'warmup_bias_lr': 0.1,
        'box': 7.5,
        'cls': 0.5,
        'dfl': 1.5,
        'pose': 12.0,
        'kobj': 1.0,
        'nbs': 64,
        'hsv_h': 0.015,
        'hsv_s': 0.9,
        'hsv_v': 0.6,
        'degrees': 10.0,
        'translate': 0.1,
        'scale': 0.5,
        'shear': 0.0,
        'perspective': 0.0,
        'flipud': 0.0,
        'fliplr': 0.5,
        'bgr': 0.0,
        'mosaic': 1.0,
        'mixup': 0.0,
        'copy_paste': 0.0,
        'copy_paste_mode': 'flip',
        'auto_augment': 'randaugment',
        'erasing': 0.4,
        'cfg': None,
        'tracker': 'botsort.yaml'
    }
    seed_everything(seed)

    visible_data_path = '/content/drive/My Drive/pguard/dataset.yaml'
    thermal_data_path = '/content/drive/My Drive/pguard/thermal/thermal.yaml'
    labeled_thermal_data_path = '/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled_temp.yaml'

    LOGGER.info("Validating datasets...")
    LOGGER.debug(f"kwargs before validation: {kwargs}")
    for yaml_path in [visible_data_path, thermal_data_path, labeled_thermal_data_path]:
        data_info = validate_dataset(yaml_path)
        if 'dataset.yaml' in yaml_path and len(data_info.get('train', [])) > 1000:
            kwargs['batch'] = 16
            LOGGER.info(f"Increased batch size to {kwargs['batch']} for large visible dataset")
        elif len(data_info.get('train', [])) < 100 and 'labeled' not in yaml_path:
            LOGGER.warning(f"Training with small dataset ({len(data_info.get('train', []))} images) for {yaml_path}. Performance may be suboptimal.")

    LOGGER.info("Dataset validation completed. Starting model loading...")

    drive_weights_path = '/content/drive/My Drive/pguard/runs/train/yolov8s_exp/weights/best.pt'
    local_weights_path = '/content/best.pt'
    if os.path.exists(drive_weights_path):
        LOGGER.info(f"Copying weights from {drive_weights_path} to {local_weights_path}...")
        start_copy_time = time.time()
        shutil.copyfile(drive_weights_path, local_weights_path)
        copy_time = time.time() - start_copy_time
        LOGGER.info(f"Copied weights to {local_weights_path} in {copy_time:.2f} seconds")
        file_size = os.path.getsize(local_weights_path) / (1024 * 1024)
        LOGGER.info(f"Weights file size: {file_size:.2f} MB")
        if file_size < 1:
            LOGGER.error("Weights file appears to be corrupted (size too small).")
            raise RuntimeError("Weights file appears to be corrupted.")
    else:
        LOGGER.error(f"Weights file not found at {drive_weights_path}. Cannot proceed without pre-trained weights.")
        raise FileNotFoundError(f"Weights file not found at {drive_weights_path}")

    LOGGER.info("Clearing GPU memory before model loading...")
    torch.cuda.empty_cache()

    @timeout_decorator.timeout(300)
    def load_model_with_timeout():
        start_time = time.time()
        LOGGER.info(f"Attempting to load weights from {local_weights_path}")
        model = YOLO(kwargs['model'])
        LOGGER.info("YOLOv8s model initialized. Loading weights...")
        model.load(local_weights_path)
        elapsed_time = time.time() - start_time
        LOGGER.info(f"Model loading completed in {elapsed_time:.2f} seconds")
        return model

    try:
        model = load_model_with_timeout()
    except TimeoutError:
        LOGGER.error("Model loading timed out after 5 minutes. Aborting.")
        raise TimeoutError("Model loading timed out after 5 minutes. Please check the weights file and runtime resources.")
    except Exception as e:
        LOGGER.error(f"Failed to load model: {e}")
        raise

    LOGGER.debug(f"kwargs before model.train: {kwargs}")
    custom_trainer = partial(
        CustomTrainer,
        target_domain_data_cfg=thermal_data_path,
        labeled_thermal_data=labeled_thermal_data_path
    )
    LOGGER.info("Starting visible-to-thermal training with semi-supervised learning and adversarial domain adaptation...")
    results = model.train(
        trainer=custom_trainer,
        **kwargs
    )

    train_dir = str(results.save_dir)
    save_dir_drive = "/content/drive/My Drive/pguard/runs/adapt/visible_to_thermal_semisupervised_adversarial"
    os.makedirs(save_dir_drive, exist_ok=True)
    shutil.copytree(train_dir, save_dir_drive, dirs_exist_ok=True)
    LOGGER.info(f"Training results saved to {save_dir_drive}")
    model.val(data=thermal_data_path, name='val_visible_to_thermal')

if __name__ == '__main__':
    main()

Overwriting train.py


In [ ]:
%cd /content/Domain-adaptation-object-detection-with-YOLOv8
!python train.py


/content/Domain-adaptation-object-detection-with-YOLOv8
E0000 00:00:1745059789.431054   49356 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745059789.440921   49356 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Validating datasets...
Dataset /content/drive/My Drive/pguard/dataset.yaml: train images=2973 (scanned), val images=638 (scanned)
Empty 2 label files for /content/drive/My Drive/pguard/dataset.yaml: ['/content/drive/My Drive/pguard/train/labels/imagee1230.txt', '/content/drive/My Drive/pguard/train/labels/road_262.txt']...
Training with small dataset (43 images) for /content/drive/My Drive/pguard/dataset.yaml. Performance may be suboptimal.
Dataset /content/drive/My Drive/pguard/thermal/thermal.yaml: train images=1232 (scanned), val images=465 (scanned)
Found 1232 missing label files 

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 -f https://download.pytorch.org/whl/torch_stable.html

Found existing installation: torch 2.0.1+cu118
Uninstalling torch-2.0.1+cu118:
  Successfully uninstalled torch-2.0.1+cu118
Found existing installation: torchvision 0.15.2+cu118
Uninstalling torchvision-0.15.2+cu118:
  Successfully uninstalled torchvision-0.15.2+cu118
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in links: https://download.pytorch.org/whl/torch_stable.html
  Using cached https://download.pytorch.org/whl/cu118/torch-2.0.1%2Bcu118-cp311-cp311-linux_x86_64.whl (2267.3 MB)
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.15.2%2Bcu118-cp311-cp311-linux_x86_64.whl (6.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 22.2 MB/s eta 0:00:00


In [ ]:
!rm -f "/content/drive/My Drive/pguard/train/labels.cache"
!rm -f "/content/drive/My Drive/pguard/thermal/train/labels.cache"
!rm -f "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels.cache"

In [ ]:
!ls -1 "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/labels" | wc -l

200


In [ ]:
for yaml_path in [
    '/content/drive/My Drive/pguard/thermal/thermal.yaml',
    '/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml'
]:
    cache_path = yaml_path.replace('.yaml', '.cache')
    if os.path.exists(cache_path):
        os.remove(cache_path)
        print(f"Cleared cache file: {cache_path}")

In [ ]:
!pip install torch==2.0.1 tensorboard ultralytics==8.3.111

In [ ]:
!ls -1 "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images" | wc -l

200


In [ ]:
!ls -1 "/content/drive/My Drive/pguard/thermal/train/images" | wc -l

1232


In [ ]:
yaml_content = """
path: /content/drive/My Drive/pguard/thermal/thermal_labeled
train: /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images
val: /content/drive/My Drive/pguard/thermal/thermal_labeled/val/images
nc: 1
names: ['lp']
"""
with open('/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml', 'w') as f:
    f.write(yaml_content)

In [ ]:
yaml_content = """
path: /content/drive/My Drive/pguard/thermal
train: /content/drive/My Drive/pguard/thermal/train/images
val: /content/drive/My Drive/pguard/thermal/val/images
nc: 1
names: ['lp']
"""
with open('/content/drive/My Drive/pguard/thermal/thermal.yaml', 'w') as f:
    f.write(yaml_content)

In [ ]:
# List files in both directories and find overlapping images
!ls "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images" > labeled_thermal.txt
!ls "/content/drive/My Drive/pguard/thermal/val/images" > thermal_val.txt
!comm -12 labeled_thermal.txt thermal_val.txt > overlapping_files.txt

# Create a directory for overlapping images
!mkdir -p "/content/drive/My Drive/pguard/thermal/overlapping_images"

# Move overlapping images and their labels from the validation set to the overlapping_images directory
with open("overlapping_files.txt", "r") as f:
    for file in f.read().splitlines():
        !mv "/content/drive/My Drive/pguard/thermal/val/images/{file}" "/content/drive/My Drive/pguard/thermal/overlapping_images/"
        label_file = file.rsplit(".", 1)[0] + ".txt"
        !mv "/content/drive/My Drive/pguard/thermal/val/labels/{label_file}" "/content/drive/My Drive/pguard/thermal/overlapping_images/"



# Clear the validation cache to force a re-scan
!rm "/content/drive/My Drive/pguard/thermal/val/labels.cache"

In [ ]:
!cat "/content/drive/My Drive/pguard/thermal/thermal.yaml"


path: /content/drive/My Drive/pguard/thermal
train: /content/drive/My Drive/pguard/thermal/train/images
val: /content/drive/My Drive/pguard/thermal/val/images
nc: 1
names: ['lp']


In [ ]:

!rm "/content/drive/My Drive/pguard/thermal/train/labels.cache"
!rm "/content/drive/My Drive/pguard/thermal/val/labels.cache"

rm: cannot remove '/content/drive/My Drive/pguard/thermal/val/labels.cache': No such file or directory


In [ ]:
!cat "/content/drive/My Drive/pguard/thermal/thermal.yaml"


path: /content/drive/My Drive/pguard/thermal
train: /content/drive/My Drive/pguard/thermal/train/images
val: /content/drive/My Drive/pguard/thermal/val/images
nc: 1
names: ['lp']


In [ ]:
thermal_yaml_path = "/content/drive/My Drive/pguard/thermal/thermal.yaml"

yaml_content = """
train: /content/drive/My Drive/pguard/thermal/val/images
val: /content/drive/My Drive/pguard/thermal/val/images
nc: 1
names: ['lp']
"""

with open(thermal_yaml_path, "w") as f:
    f.write(yaml_content.strip())

print("✅ thermal.yaml file updated!")



✅ thermal.yaml file updated!


In [ ]:
import shutil
import os

# Source directory (local)
source_dir = "runs/detect/visible_to_thermal_semisupervised8"

# Destination directory (Google Drive)
dest_dir = "/content/drive/My Drive/pguard/runs/detect/visible_to_thermal_semisupervised8"

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Copy the entire directory to Drive
shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)

print(f"Results copied to {dest_dir}")

Results copied to /content/drive/My Drive/pguard/runs/detect/visible_to_thermal_semisupervised8


In [ ]:
import os

save_dir = "/content/runs/detect/visible_to_thermal_semisupervised10"
if os.path.exists(save_dir):
    print("Save directory exists:", save_dir)
    print("Contents:")
    for file in os.listdir(save_dir):
        print(f"- {file}")
else:
    print("Save directory not found:", save_dir)

Save directory not found: /content/runs/detect/visible_to_thermal_semisupervised10


In [ ]:
!pip install ultralytics timeout-decorator -q

  Preparing metadata (setup.py) ... done


In [ ]:
!cat "/content/drive/My Drive/pguard/thermal/thermal_labeled/thermal_labeled.yaml"


path: /content/drive/My Drive/pguard/thermal/thermal_labeled
train: /content/drive/My Drive/pguard/thermal/thermal_labeled/train/images
val: /content/drive/My Drive/pguard/thermal/thermal_labeled/val/images
nc: 1
names: ['lp']


In [ ]:
!cat "/content/drive/My Drive/pguard/dataset.yaml"

path: /content/drive/My Drive/pguard
train: train/images
val: val/images
nc: 1
names: ['lp']


In [ ]:
ls -l "/content/drive/My Drive/pguard/thermal/thermal_labeled/train/images" | grep -E '\.png|\.jpg|\.jpeg|\.bmp' | wc -l

200


In [ ]:
ls -l "/content/drive/My Drive/pguard/thermal/val/labels" | wc -l

1700


In [ ]:
ls -l "/content/drive/My Drive/pguard/train/images" | grep -E '\.png|\.jpg|\.jpeg|\.bmp' | wc -l

2973
